### There was a few corrections in the dataset that needed to be done after geetiing the mail back from the data providers and that is the whole purpose of this file.

## I had found that there were some errors in the names that would make it into a different projet and after confirmation from the data providers. So I am going to fix that first.

In [1]:
# ============================================================
# Step 1 - Cell 1
# Imports and file paths
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


# Immutable corrected transaction dataset
TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


# Main controlled product mapping created in this step
MASTER_MAPPING_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling.csv"
)


# Audit of automatically applied spelling corrections
SPELLING_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_product_spelling_corrections_step1_audit.csv"
)


# Names that require human confirmation
MANUAL_REVIEW_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_product_name_manual_review_candidates_step1.csv"
)


DATA_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


if not TRANSACTION_INPUT_FILE.exists():
    raise FileNotFoundError(
        "Corrected transaction dataset was not found:\n"
        f"{TRANSACTION_INPUT_FILE}"
    )


print("Input transaction dataset:")
print(TRANSACTION_INPUT_FILE)

print()
print("Master mapping output:")
print(MASTER_MAPPING_OUTPUT_FILE)

Input transaction dataset:
eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv

Master mapping output:
eden_datasets/UL_EDEN_master_product_mapping_step1_spelling.csv


In [2]:
# ============================================================
# Step 1 - Cell 2
# Load and validate immutable corrected transactions
# ============================================================

transactions_step1 = pd.read_csv(
    TRANSACTION_INPUT_FILE
)


required_columns = [
    "TransDate",
    "TransValue",
    "PLUName",
    "GroupCode",
    "GroupName",
    "PLUCode",
    "Date",
    "TransactionID",
    "UnitSold",
]


missing_columns = [
    column
    for column in required_columns
    if column not in transactions_step1.columns
]


if missing_columns:
    raise ValueError(
        "Required transaction columns are missing:\n"
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# Safe type conversion
# ------------------------------------------------------------

transactions_step1["TransDate"] = pd.to_datetime(
    transactions_step1["TransDate"],
    errors="raise"
)


transactions_step1["Date"] = pd.to_datetime(
    transactions_step1["Date"],
    errors="raise"
).dt.normalize()


transactions_step1["PLUCode"] = (
    pd.to_numeric(
        transactions_step1["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


transactions_step1["GroupCode"] = (
    pd.to_numeric(
        transactions_step1["GroupCode"],
        errors="raise"
    )
    .astype("int64")
)


transactions_step1["UnitSold"] = (
    pd.to_numeric(
        transactions_step1["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


transactions_step1["TransValue"] = pd.to_numeric(
    transactions_step1["TransValue"],
    errors="raise"
)


# Preserve exact original names
transactions_step1["PLUName"] = (
    transactions_step1["PLUName"]
    .astype("string")
    .str.strip()
)


transactions_step1["GroupName"] = (
    transactions_step1["GroupName"]
    .astype("string")
    .str.strip()
)


# ------------------------------------------------------------
# Record immutable totals
# ------------------------------------------------------------

step1_original_row_count = len(
    transactions_step1
)


step1_original_unit_total = int(
    transactions_step1["UnitSold"].sum()
)


step1_original_value_cents = int(
    (
        transactions_step1["TransValue"]
        * 100
    )
    .round()
    .sum()
)


# ------------------------------------------------------------
# Validate exact source dataset
# ------------------------------------------------------------

assert step1_original_row_count == 138_983, (
    "Expected 138,983 transaction rows."
)


assert step1_original_unit_total == 141_480, (
    "Expected corrected UnitSold total of 141,480."
)


assert transactions_step1["PLUCode"].nunique() == 236, (
    "Expected 236 PLU codes including OPEN UL."
)


assert transactions_step1["UnitSold"].ge(1).all(), (
    "UnitSold contains a value below 1."
)


print("Immutable transaction dataset validated.")
print()
print(
    "Rows:",
    f"{step1_original_row_count:,}"
)
print(
    "PLU codes:",
    f"{transactions_step1['PLUCode'].nunique():,}"
)
print(
    "Corrected unit total:",
    f"{step1_original_unit_total:,}"
)

Immutable transaction dataset validated.

Rows: 138,983
PLU codes: 236
Corrected unit total: 141,480


In [3]:
# ============================================================
# Step 1 - Cell 3
# Create one base mapping row per PLUCode
# ============================================================

plu_consistency_check = (
    transactions_step1
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        ProductNameCount=(
            "PLUName",
            "nunique"
        ),
        GroupCodeCount=(
            "GroupCode",
            "nunique"
        ),
        GroupNameCount=(
            "GroupName",
            "nunique"
        ),
    )
)


inconsistent_plu_codes = (
    plu_consistency_check.loc[
        (
            plu_consistency_check[
                "ProductNameCount"
            ].ne(1)
            |
            plu_consistency_check[
                "GroupCodeCount"
            ].ne(1)
            |
            plu_consistency_check[
                "GroupNameCount"
            ].ne(1)
        )
    ]
)


if len(inconsistent_plu_codes) > 0:
    display(inconsistent_plu_codes)

    raise ValueError(
        "At least one PLU code has multiple names or groups. "
        "The mapping cannot be created safely."
    )


# ------------------------------------------------------------
# Create one row per PLUCode
# ------------------------------------------------------------

master_product_mapping = (
    transactions_step1
    .sort_values(
        [
            "PLUCode",
            "Date",
            "TransDate",
        ]
    )
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        PLUName_Original=(
            "PLUName",
            "first"
        ),
        GroupCode=(
            "GroupCode",
            "first"
        ),
        GroupName=(
            "GroupName",
            "first"
        ),
        FirstObservedDate=(
            "Date",
            "min"
        ),
        LastObservedDate=(
            "Date",
            "max"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
        TotalTransactionValue=(
            "TransValue",
            "sum"
        ),
    )
)


# ------------------------------------------------------------
# Basic text standardisation only
# ------------------------------------------------------------
# Original name is preserved exactly.
# Corrected name is upper-case with repeated spaces removed.

master_product_mapping[
    "PLUName_Corrected"
] = (
    master_product_mapping[
        "PLUName_Original"
    ]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.upper()
)


# ------------------------------------------------------------
# Every PLU initially remains a separate product
# ------------------------------------------------------------

master_product_mapping[
    "CanonicalProductID"
] = (
    "PLU_"
    + master_product_mapping[
        "PLUCode"
    ].astype(str)
)


master_product_mapping[
    "CanonicalProductName"
] = master_product_mapping[
    "PLUName_Corrected"
]


# These fields will be completed in later controlled steps
master_product_mapping[
    "SupplierBrand"
] = pd.NA

master_product_mapping[
    "ProductFamily"
] = pd.NA

master_product_mapping[
    "PriceTier"
] = pd.NA


master_product_mapping[
    "MappingType"
] = "UNCHANGED"


master_product_mapping[
    "MappingReason"
] = (
    "No approved spelling correction required in Step 1."
)


master_product_mapping[
    "MappingConfidence"
] = "NOT_APPLICABLE"


master_product_mapping[
    "MappingStatus"
] = "APPROVED_STEP_1"


master_product_mapping[
    "MappingVersion"
] = "STEP_1_SPELLING_V1"


# OPEN UL remains in the mapping for audit,
# but it is explicitly excluded from forecasting.
open_ul_mapping_mask = (
    master_product_mapping[
        "PLUName_Original"
    ]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


assert int(open_ul_mapping_mask.sum()) == 1


master_product_mapping.loc[
    open_ul_mapping_mask,
    "MappingStatus"
] = "EXCLUDE_FROM_FORECASTING"


master_product_mapping.loc[
    open_ul_mapping_mask,
    "MappingReason"
] = (
    "Open-value POS entry; not a forecastable product."
)


print("Base master mapping created.")
print()
print(
    "Mapping rows:",
    len(master_product_mapping)
)
print(
    "Unique temporary canonical IDs:",
    master_product_mapping[
        "CanonicalProductID"
    ].nunique()
)

Base master mapping created.

Mapping rows: 236
Unique temporary canonical IDs: 236


In [4]:
# ============================================================
# Step 1 - Cell 4
# Apply approved high-confidence spelling corrections
# ============================================================

approved_spelling_corrections = [
    {
        "PLUCode": 42552,
        "ExpectedOriginal": "ALERNATIVE MILK",
        "CorrectedName": "ALTERNATIVE MILK",
        "Reason": "Corrected ALERNATIVE to ALTERNATIVE.",
    },
    {
        "PLUCode": 4241502,
        "ExpectedOriginal": "ALUXARY CROSSANT / PASTERIES",
        "CorrectedName": "LUXURY CROISSANT / PASTRIES",
        "Reason": (
            "Corrected ALUXARY, CROSSANT and PASTERIES."
        ),
    },
    {
        "PLUCode": 4241536,
        "ExpectedOriginal": "BREAKSFAST POTS",
        "CorrectedName": "BREAKFAST POTS",
        "Reason": "Corrected BREAKSFAST to BREAKFAST.",
    },
    {
        "PLUCode": 42527103,
        "ExpectedOriginal": "BEWLEYS CAPPUCINO",
        "CorrectedName": "BEWLEYS CAPPUCCINO",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4241460,
        "ExpectedOriginal": "CAPPUCINO 12OZ",
        "CorrectedName": "CAPPUCCINO 12OZ",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4241461,
        "ExpectedOriginal": "CAPPUCINO 16OZ",
        "CorrectedName": "CAPPUCCINO 16OZ",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4251511,
        "ExpectedOriginal": "CAPPUCINO SM",
        "CorrectedName": "CAPPUCCINO SM",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4251512,
        "ExpectedOriginal": "CAPPUCINO MED",
        "CorrectedName": "CAPPUCCINO MED",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4251513,
        "ExpectedOriginal": "CAPPUCINO LG",
        "CorrectedName": "CAPPUCCINO LG",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4251505,
        "ExpectedOriginal": "CARAMEL MACHIATO SM",
        "CorrectedName": "CARAMEL MACCHIATO SM",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4251506,
        "ExpectedOriginal": "CARAMEL MACHIATO MED",
        "CorrectedName": "CARAMEL MACCHIATO MED",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4251507,
        "ExpectedOriginal": "CARAMEL MACHIATO LG",
        "CorrectedName": "CARAMEL MACCHIATO LG",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4251520,
        "ExpectedOriginal": "ICED CARAMEL MACHIATO MED",
        "CorrectedName": "ICED CARAMEL MACCHIATO MED",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4251521,
        "ExpectedOriginal": "ICED CARAMEL MACHIATO LG",
        "CorrectedName": "ICED CARAMEL MACCHIATO LG",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4241466,
        "ExpectedOriginal": "CHIA 12 OZ",
        "CorrectedName": "CHAI 12 OZ",
        "Reason": (
            "Corrected CHIA to CHAI; this is a hot beverage "
            "and corresponds with CHAI 16OZ."
        ),
    },
    {
        "PLUCode": 42543,
        "ExpectedOriginal": "DELLI SANDWICH - NO MEAT",
        "CorrectedName": "DELI SANDWICH - NO MEAT",
        "Reason": "Corrected DELLI to DELI.",
    },
    {
        "PLUCode": 42544,
        "ExpectedOriginal": (
            "DELLI SANDWICH X3 SALAD X1 MEAT"
        ),
        "CorrectedName": (
            "DELI SANDWICH X3 SALAD X1 MEAT"
        ),
        "Reason": "Corrected DELLI to DELI.",
    },
    {
        "PLUCode": 425992,
        "ExpectedOriginal": "MOCA",
        "CorrectedName": "MOCHA",
        "Reason": "Corrected MOCA to MOCHA.",
    },
    {
        "PLUCode": 42571,
        "ExpectedOriginal": (
            "PROTIAN BAR POWER BALLS CT"
        ),
        "CorrectedName": (
            "PROTEIN BAR POWER BALLS CT"
        ),
        "Reason": "Corrected PROTIAN to PROTEIN.",
    },
    {
        "PLUCode": 42555,
        "ExpectedOriginal": (
            "R0OASTED NOTES EXPRESSO CT"
        ),
        "CorrectedName": (
            "ROASTED NOTES ESPRESSO CT"
        ),
        "Reason": (
            "Corrected zero in R0OASTED and "
            "EXPRESSO to ESPRESSO."
        ),
    },
    {
        "PLUCode": 4241422,
        "ExpectedOriginal": (
            "SALTED CARAMEL PEANUR BISCUIT CAKE"
        ),
        "CorrectedName": (
            "SALTED CARAMEL PEANUT BISCUIT CAKE"
        ),
        "Reason": "Corrected PEANUR to PEANUT.",
    },
    {
        "PLUCode": 4241448,
        "ExpectedOriginal": "SINGEL ESPESSO",
        "CorrectedName": "SINGLE ESPRESSO",
        "Reason": (
            "Corrected SINGEL and ESPESSO."
        ),
    },
    {
        "PLUCode": 4241446,
        "ExpectedOriginal": (
            "SPECIALILITY TEA 12 OZ"
        ),
        "CorrectedName": (
            "SPECIALITY TEA 12 OZ"
        ),
        "Reason": (
            "Corrected SPECIALILITY to SPECIALITY."
        ),
    },
    {
        "PLUCode": 2000000130,
        "ExpectedOriginal": (
            "JUICE 200 ML 200ML BTL"
        ),
        "CorrectedName": "JUICE 200ML BTL",
        "Reason": (
            "Removed the duplicated 200 ML description."
        ),
    },
    {
        "PLUCode": 4251515,
        "ExpectedOriginal": "CARAMEL FRAPP LG",
        "CorrectedName": "CARAMEL FRAP LG",
        "Reason": (
            "Standardised FRAPP to FRAP to match "
            "the associated medium product."
        ),
    },
]


approved_spelling_corrections_df = pd.DataFrame(
    approved_spelling_corrections
)


assert approved_spelling_corrections_df[
    "PLUCode"
].is_unique, (
    "A PLU code appears more than once in the approved "
    "spelling-correction list."
)


# ------------------------------------------------------------
# Apply each correction safely
# ------------------------------------------------------------

for correction in approved_spelling_corrections:

    plu_code = correction["PLUCode"]

    mapping_mask = (
        master_product_mapping[
            "PLUCode"
        ].eq(plu_code)
    )


    assert int(mapping_mask.sum()) == 1, (
        f"Expected exactly one mapping row for PLU {plu_code}."
    )


    actual_original = (
        master_product_mapping.loc[
            mapping_mask,
            "PLUName_Original"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    expected_original = (
        correction["ExpectedOriginal"]
        .strip()
        .upper()
    )


    assert actual_original == expected_original, (
        f"Original-name mismatch for PLU {plu_code}.\n"
        f"Expected: {expected_original}\n"
        f"Found: {actual_original}"
    )


    corrected_name = (
        correction["CorrectedName"]
        .strip()
        .upper()
    )


    master_product_mapping.loc[
        mapping_mask,
        "PLUName_Corrected"
    ] = corrected_name


    master_product_mapping.loc[
        mapping_mask,
        "CanonicalProductName"
    ] = corrected_name


    master_product_mapping.loc[
        mapping_mask,
        "MappingType"
    ] = "SPELLING_CORRECTION"


    master_product_mapping.loc[
        mapping_mask,
        "MappingReason"
    ] = correction["Reason"]


    master_product_mapping.loc[
        mapping_mask,
        "MappingConfidence"
    ] = "HIGH"


    master_product_mapping.loc[
        mapping_mask,
        "MappingStatus"
    ] = "APPROVED_STEP_1"


print(
    "Approved spelling corrections applied:",
    len(approved_spelling_corrections_df)
)

Approved spelling corrections applied: 25


In [5]:
# ============================================================
# Step 1 - Cell 5
# Create manual-review candidates without changing names
# ============================================================

manual_review_candidates = [
    {
        "PLUCode": 4241404,
        "ExpectedOriginal": "ODONNELL /POPCORN",
        "SuggestedName": "O'DONNELL'S POPCORN",
        "PotentialIssue": (
            "Possible missing apostrophe, possessive S "
            "and incorrect separator."
        ),
        "ReviewConfidence": "MEDIUM",
    },
    {
        "PLUCode": 2024241,
        "ExpectedOriginal": "JOS POWERBALL",
        "SuggestedName": "JO'S POWERBALL",
        "PotentialIssue": (
            "Possible missing apostrophe; could also be "
            "a supplier-specific brand name."
        ),
        "ReviewConfidence": "LOW",
    },
    {
        "PLUCode": 2000000269,
        "ExpectedOriginal": (
            "BALLYGOWAN SPARK WATER 330ML"
        ),
        "SuggestedName": (
            "BALLYGOWAN SPARKLING WATER 330ML"
        ),
        "PotentialIssue": (
            "SPARK may be an abbreviation for SPARKLING."
        ),
        "ReviewConfidence": "MEDIUM",
    },
    {
        "PLUCode": 4241537,
        "ExpectedOriginal": "CRUSTED CROISSANTS",
        "SuggestedName": "REQUIRES CONFIRMATION",
        "PotentialIssue": (
            "The intended product may be a specific type "
            "of crusted or filled croissant."
        ),
        "ReviewConfidence": "LOW",
    },
    {
        "PLUCode": 4241455,
        "ExpectedOriginal": "ICED VANILLA 16OZ",
        "SuggestedName": "ICED VANILLA LATTE 16OZ",
        "PotentialIssue": (
            "LATTE may be missing, but the transaction data "
            "alone cannot confirm this."
        ),
        "ReviewConfidence": "LOW",
    },
    {
        "PLUCode": 4241405,
        "ExpectedOriginal": "HUNKY DORY",
        "SuggestedName": "HUNKY DORYS",
        "PotentialIssue": (
            "Possible shortened brand name."
        ),
        "ReviewConfidence": "MEDIUM",
    },
    {
        "PLUCode": 4241409,
        "ExpectedOriginal": (
            "PACKS OF SWEET - WINE GUMS"
        ),
        "SuggestedName": (
            "PACK OF SWEETS - WINE GUMS"
        ),
        "PotentialIssue": (
            "Possible singular/plural wording error."
        ),
        "ReviewConfidence": "LOW",
    },
    {
        "PLUCode": 4241415,
        "ExpectedOriginal": (
            "LUXURY MILLIONAIRES FLAPJACK"
        ),
        "SuggestedName": (
            "LUXURY MILLIONAIRE'S FLAPJACK"
        ),
        "PotentialIssue": (
            "Possible possessive punctuation correction."
        ),
        "ReviewConfidence": "MEDIUM",
    },
]


manual_review_candidates_df = pd.DataFrame(
    manual_review_candidates
)


assert manual_review_candidates_df[
    "PLUCode"
].is_unique


# Ensure manual-review candidates do not overlap with
# approved spelling corrections.
overlapping_review_codes = set(
    manual_review_candidates_df["PLUCode"]
).intersection(
    set(
        approved_spelling_corrections_df["PLUCode"]
    )
)


assert not overlapping_review_codes, (
    "Manual-review and approved corrections overlap:\n"
    f"{sorted(overlapping_review_codes)}"
)


# ------------------------------------------------------------
# Validate and mark manual-review rows
# ------------------------------------------------------------

for candidate in manual_review_candidates:

    plu_code = candidate["PLUCode"]

    mapping_mask = (
        master_product_mapping[
            "PLUCode"
        ].eq(plu_code)
    )


    assert int(mapping_mask.sum()) == 1, (
        f"Expected one mapping row for PLU {plu_code}."
    )


    actual_original = (
        master_product_mapping.loc[
            mapping_mask,
            "PLUName_Original"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    expected_original = (
        candidate["ExpectedOriginal"]
        .strip()
        .upper()
    )


    assert actual_original == expected_original, (
        f"Manual-review name mismatch for PLU {plu_code}."
    )


    # Important:
    # PLUName_Corrected and CanonicalProductName are not changed.
    master_product_mapping.loc[
        mapping_mask,
        "MappingType"
    ] = "MANUAL_REVIEW"


    master_product_mapping.loc[
        mapping_mask,
        "MappingReason"
    ] = candidate["PotentialIssue"]


    master_product_mapping.loc[
        mapping_mask,
        "MappingConfidence"
    ] = candidate["ReviewConfidence"]


    master_product_mapping.loc[
        mapping_mask,
        "MappingStatus"
    ] = "PENDING_MANUAL_REVIEW"


print(
    "Manual-review candidates identified:",
    len(manual_review_candidates_df)
)

display(manual_review_candidates_df)

Manual-review candidates identified: 8


,PLUCode,ExpectedOriginal,SuggestedName,PotentialIssue,ReviewConfidence
0,4241404,ODONNELL /POPCORN,O'DONNELL'S POPCORN,"Possible missing apostrophe, possessive S and ...",MEDIUM
1,2024241,JOS POWERBALL,JO'S POWERBALL,Possible missing apostrophe; could also be a s...,LOW
2,2000000269,BALLYGOWAN SPARK WATER 330ML,BALLYGOWAN SPARKLING WATER 330ML,SPARK may be an abbreviation for SPARKLING.,MEDIUM
3,4241537,CRUSTED CROISSANTS,REQUIRES CONFIRMATION,The intended product may be a specific type of...,LOW
4,4241455,ICED VANILLA 16OZ,ICED VANILLA LATTE 16OZ,"LATTE may be missing, but the transaction data...",LOW
5,4241405,HUNKY DORY,HUNKY DORYS,Possible shortened brand name.,MEDIUM
6,4241409,PACKS OF SWEET - WINE GUMS,PACK OF SWEETS - WINE GUMS,Possible singular/plural wording error.,LOW
7,4241415,LUXURY MILLIONAIRES FLAPJACK,LUXURY MILLIONAIRE'S FLAPJACK,Possible possessive punctuation correction.,MEDIUM


In [6]:
# ============================================================
# Step 1 - Cell 6
# Validate Step 1 isolation and integrity
# ============================================================

# ------------------------------------------------------------
# Mapping-table integrity
# ------------------------------------------------------------

assert len(master_product_mapping) == 236, (
    "Master mapping must contain 236 PLU rows."
)


assert master_product_mapping[
    "PLUCode"
].is_unique, (
    "PLUCode is not unique in the master mapping."
)


assert master_product_mapping[
    "CanonicalProductID"
].is_unique, (
    "A product merge has accidentally occurred in Step 1."
)


assert master_product_mapping[
    "CanonicalProductID"
].nunique() == 236, (
    "Every PLU must still have its own canonical ID."
)


assert master_product_mapping[
    "PLUName_Original"
].notna().all()


assert master_product_mapping[
    "PLUName_Corrected"
].notna().all()


assert master_product_mapping[
    "CanonicalProductName"
].notna().all()


assert int(
    master_product_mapping[
        "MappingType"
    ].eq("SPELLING_CORRECTION").sum()
) == len(approved_spelling_corrections_df)


assert int(
    master_product_mapping[
        "MappingStatus"
    ].eq("PENDING_MANUAL_REVIEW").sum()
) == len(manual_review_candidates_df)


# ------------------------------------------------------------
# KIMBOX and KIMBOCK must remain separate
# ------------------------------------------------------------

kimbox_mapping = master_product_mapping.loc[
    master_product_mapping[
        "PLUName_Corrected"
    ].str.contains(
        "KIMBOX",
        case=False,
        na=False
    )
]


kimbock_mapping = master_product_mapping.loc[
    master_product_mapping[
        "PLUName_Corrected"
    ].str.contains(
        "KIMBOCK",
        case=False,
        na=False
    )
]


assert len(kimbox_mapping) > 0
assert len(kimbock_mapping) > 0


assert set(
    kimbox_mapping["CanonicalProductID"]
).isdisjoint(
    set(
        kimbock_mapping["CanonicalProductID"]
    )
), (
    "KIMBOX and KIMBOCK have been accidentally merged."
)


# ------------------------------------------------------------
# Temporarily join the mapping for validation only
# ------------------------------------------------------------

step1_validation_join = transactions_step1.merge(
    master_product_mapping[
        [
            "PLUCode",
            "PLUName_Original",
            "PLUName_Corrected",
            "CanonicalProductID",
            "CanonicalProductName",
            "MappingStatus",
        ]
    ],
    on="PLUCode",
    how="left",
    validate="many_to_one"
)


assert len(step1_validation_join) == (
    step1_original_row_count
), (
    "Joining the mapping changed the transaction row count."
)


assert step1_validation_join[
    "CanonicalProductID"
].notna().all(), (
    "At least one transaction has no product mapping."
)


assert int(
    step1_validation_join["UnitSold"].sum()
) == step1_original_unit_total, (
    "Joining the mapping changed the UnitSold total."
)


joined_value_cents = int(
    (
        step1_validation_join["TransValue"]
        * 100
    )
    .round()
    .sum()
)


assert joined_value_cents == (
    step1_original_value_cents
), (
    "Joining the mapping changed transaction value."
)


# ------------------------------------------------------------
# Create spelling-correction audit
# ------------------------------------------------------------

spelling_correction_audit = (
    master_product_mapping.loc[
        master_product_mapping[
            "MappingType"
        ].eq("SPELLING_CORRECTION"),
        [
            "PLUCode",
            "PLUName_Original",
            "PLUName_Corrected",
            "GroupCode",
            "GroupName",
            "FirstObservedDate",
            "LastObservedDate",
            "TransactionRows",
            "TotalUnits",
            "MappingReason",
            "MappingConfidence",
            "MappingStatus",
        ],
    ]
    .sort_values(
        "PLUName_Original"
    )
    .reset_index(drop=True)
)


print("Step 1 integrity checks passed.")
print()
print(
    "Mapping rows:",
    len(master_product_mapping)
)
print(
    "Unique canonical IDs:",
    master_product_mapping[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Approved spelling corrections:",
    len(spelling_correction_audit)
)
print(
    "Pending manual reviews:",
    len(manual_review_candidates_df)
)
print(
    "Transaction rows unchanged:",
    f"{len(step1_validation_join):,}"
)
print(
    "UnitSold total unchanged:",
    f"{int(step1_validation_join['UnitSold'].sum()):,}"
)

print()
print("Mapping status summary:")

display(
    master_product_mapping[
        "MappingStatus"
    ]
    .value_counts()
    .rename_axis("MappingStatus")
    .reset_index(name="Products")
)

print()
print("Approved spelling corrections:")

display(spelling_correction_audit)

Step 1 integrity checks passed.

Mapping rows: 236
Unique canonical IDs: 236
Approved spelling corrections: 25
Pending manual reviews: 8
Transaction rows unchanged: 138,983
UnitSold total unchanged: 141,480

Mapping status summary:


,MappingStatus,Products
0,APPROVED_STEP_1,227
1,PENDING_MANUAL_REVIEW,8
2,EXCLUDE_FROM_FORECASTING,1



Approved spelling corrections:


,PLUCode,PLUName_Original,PLUName_Corrected,GroupCode,GroupName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,MappingReason,MappingConfidence,MappingStatus
0,42552,ALERNATIVE MILK,ALTERNATIVE MILK,1,HOT BEVS,2025-10-14,2026-03-30,839,839,Corrected ALERNATIVE to ALTERNATIVE.,HIGH,APPROVED_STEP_1
1,4241502,ALUXARY CROSSANT / PASTERIES,LUXURY CROISSANT / PASTRIES,4,BREAKFAST,2025-10-14,2026-03-30,654,654,"Corrected ALUXARY, CROSSANT and PASTERIES.",HIGH,APPROVED_STEP_1
2,42527103,BEWLEYS CAPPUCINO,BEWLEYS CAPPUCCINO,1,HOT BEVS,2025-04-01,2025-10-13,517,517,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
3,4241536,BREAKSFAST POTS,BREAKFAST POTS,4,BREAKFAST,2025-04-01,2026-03-30,334,334,Corrected BREAKSFAST to BREAKFAST.,HIGH,APPROVED_STEP_1
4,4241460,CAPPUCINO 12OZ,CAPPUCCINO 12OZ,1,HOT BEVS,2025-04-01,2025-10-10,471,471,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
5,4241461,CAPPUCINO 16OZ,CAPPUCCINO 16OZ,1,HOT BEVS,2025-04-01,2025-08-29,179,179,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
6,4251513,CAPPUCINO LG,CAPPUCCINO LG,1,HOT BEVS,2025-04-02,2025-08-29,126,126,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
7,4251512,CAPPUCINO MED,CAPPUCCINO MED,1,HOT BEVS,2025-04-02,2025-09-03,189,189,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
8,4251511,CAPPUCINO SM,CAPPUCCINO SM,1,HOT BEVS,2025-04-01,2025-09-03,151,151,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
9,4251515,CARAMEL FRAPP LG,CARAMEL FRAP LG,1,HOT BEVS,2025-04-01,2025-08-29,160,160,Standardised FRAPP to FRAP to match the associ...,HIGH,APPROVED_STEP_1


In [7]:
# ============================================================
# Step 1 - Cell 7
# Arrange and save controlled Step 1 outputs
# ============================================================

master_mapping_columns = [
    "PLUCode",
    "PLUName_Original",
    "PLUName_Corrected",
    "CanonicalProductID",
    "CanonicalProductName",
    "SupplierBrand",
    "ProductFamily",
    "PriceTier",
    "GroupCode",
    "GroupName",
    "FirstObservedDate",
    "LastObservedDate",
    "TransactionRows",
    "TotalUnits",
    "TotalTransactionValue",
    "MappingType",
    "MappingReason",
    "MappingConfidence",
    "MappingStatus",
    "MappingVersion",
]


master_product_mapping = (
    master_product_mapping[
        master_mapping_columns
    ]
    .sort_values(
        [
            "PLUName_Corrected",
            "PLUCode",
        ]
    )
    .reset_index(drop=True)
)


# Add observed product information to the manual-review file
manual_review_output = (
    manual_review_candidates_df
    .merge(
        master_product_mapping[
            [
                "PLUCode",
                "PLUName_Original",
                "GroupCode",
                "GroupName",
                "FirstObservedDate",
                "LastObservedDate",
                "TransactionRows",
                "TotalUnits",
            ]
        ],
        on="PLUCode",
        how="left",
        validate="one_to_one"
    )
)


manual_review_output = manual_review_output[
    [
        "PLUCode",
        "PLUName_Original",
        "SuggestedName",
        "GroupCode",
        "GroupName",
        "FirstObservedDate",
        "LastObservedDate",
        "TransactionRows",
        "TotalUnits",
        "PotentialIssue",
        "ReviewConfidence",
    ]
]


# ------------------------------------------------------------
# Save only mapping and audit files
# The transaction dataset is not modified or resaved.
# ------------------------------------------------------------

master_product_mapping.to_csv(
    MASTER_MAPPING_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


spelling_correction_audit.to_csv(
    SPELLING_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


manual_review_output.to_csv(
    MANUAL_REVIEW_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("Step 1 files saved successfully.")
print()

print("1. Master product mapping:")
print(MASTER_MAPPING_OUTPUT_FILE)

print()
print("2. Approved spelling-correction audit:")
print(SPELLING_AUDIT_OUTPUT_FILE)

print()
print("3. Manual-review candidates:")
print(MANUAL_REVIEW_OUTPUT_FILE)

Step 1 files saved successfully.

1. Master product mapping:
eden_datasets/UL_EDEN_master_product_mapping_step1_spelling.csv

2. Approved spelling-correction audit:
eden_datasets/UL_EDEN_product_spelling_corrections_step1_audit.csv

3. Manual-review candidates:
eden_datasets/UL_EDEN_product_name_manual_review_candidates_step1.csv


In [8]:
# ============================================================
# Step 1 - Cell 8
# Reload and validate saved Step 1 mapping
# ============================================================

saved_master_mapping = pd.read_csv(
    MASTER_MAPPING_OUTPUT_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


saved_master_mapping["PLUCode"] = (
    pd.to_numeric(
        saved_master_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


saved_master_mapping["TotalUnits"] = (
    pd.to_numeric(
        saved_master_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


assert len(saved_master_mapping) == 236


assert saved_master_mapping[
    "PLUCode"
].is_unique


assert saved_master_mapping[
    "CanonicalProductID"
].is_unique, (
    "The saved Step 1 mapping contains an accidental merge."
)


assert saved_master_mapping[
    "CanonicalProductID"
].nunique() == 236


assert int(
    saved_master_mapping["TotalUnits"].sum()
) == 141_480


assert int(
    saved_master_mapping[
        "MappingType"
    ].eq("SPELLING_CORRECTION").sum()
) == 25


assert int(
    saved_master_mapping[
        "MappingStatus"
    ].eq("PENDING_MANUAL_REVIEW").sum()
) == 8


saved_open_ul = saved_master_mapping.loc[
    saved_master_mapping[
        "PLUName_Original"
    ].astype("string").str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
]


assert len(saved_open_ul) == 1


assert (
    saved_open_ul[
        "MappingStatus"
    ].iloc[0]
    == "EXCLUDE_FROM_FORECASTING"
)


print("=" * 72)
print("STEP 1 MASTER PRODUCT MAPPING COMPLETED")
print("=" * 72)
print()
print(
    "PLU mapping rows:",
    len(saved_master_mapping)
)
print(
    "Approved spelling corrections:",
    int(
        saved_master_mapping[
            "MappingType"
        ].eq("SPELLING_CORRECTION").sum()
    )
)
print(
    "Pending manual reviews:",
    int(
        saved_master_mapping[
            "MappingStatus"
        ].eq("PENDING_MANUAL_REVIEW").sum()
    )
)
print(
    "Accidental product merges:",
    236 - saved_master_mapping[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Mapped total units:",
    f"{int(saved_master_mapping['TotalUnits'].sum()):,}"
)

display(
    saved_master_mapping.loc[
        saved_master_mapping[
            "MappingType"
        ].ne("UNCHANGED")
    ]
)

STEP 1 MASTER PRODUCT MAPPING COMPLETED

PLU mapping rows: 236
Approved spelling corrections: 25
Pending manual reviews: 8
Accidental product merges: 0
Mapped total units: 141,480


,PLUCode,PLUName_Original,PLUName_Corrected,CanonicalProductID,CanonicalProductName,SupplierBrand,ProductFamily,PriceTier,GroupCode,GroupName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,TotalTransactionValue,MappingType,MappingReason,MappingConfidence,MappingStatus,MappingVersion
3,42552,ALERNATIVE MILK,ALTERNATIVE MILK,PLU_42552,ALTERNATIVE MILK,NaN,NaN,NaN,1,HOT BEVS,2025-10-14,2026-03-30,839,839,419.50,SPELLING_CORRECTION,Corrected ALERNATIVE to ALTERNATIVE.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
13,2000000269,BALLYGOWAN SPARK WATER 330ML,BALLYGOWAN SPARK WATER 330ML,PLU_2000000269,BALLYGOWAN SPARK WATER 330ML,NaN,NaN,NaN,2,COLD BEVS,2025-04-01,2025-10-02,47,47,117.50,MANUAL_REVIEW,SPARK may be an abbreviation for SPARKLING.,MEDIUM,PENDING_MANUAL_REVIEW,STEP_1_SPELLING_V1
17,42527103,BEWLEYS CAPPUCINO,BEWLEYS CAPPUCCINO,PLU_42527103,BEWLEYS CAPPUCCINO,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-10-13,517,517,1614.80,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
27,4241536,BREAKSFAST POTS,BREAKFAST POTS,PLU_4241536,BREAKFAST POTS,NaN,NaN,NaN,4,BREAKFAST,2025-04-01,2026-03-30,334,334,835.00,SPELLING_CORRECTION,Corrected BREAKSFAST to BREAKFAST.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
35,4241460,CAPPUCINO 12OZ,CAPPUCCINO 12OZ,PLU_4241460,CAPPUCCINO 12OZ,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-10-10,471,471,1799.10,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
36,4241461,CAPPUCINO 16OZ,CAPPUCCINO 16OZ,PLU_4241461,CAPPUCCINO 16OZ,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-08-29,179,179,769.70,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
37,4251513,CAPPUCINO LG,CAPPUCCINO LG,PLU_4251513,CAPPUCCINO LG,NaN,NaN,NaN,1,HOT BEVS,2025-04-02,2025-08-29,126,126,579.60,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
38,4251512,CAPPUCINO MED,CAPPUCCINO MED,PLU_4251512,CAPPUCCINO MED,NaN,NaN,NaN,1,HOT BEVS,2025-04-02,2025-09-03,189,189,812.70,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
39,4251511,CAPPUCINO SM,CAPPUCCINO SM,PLU_4251511,CAPPUCCINO SM,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-09-03,151,151,581.35,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
40,4251515,CARAMEL FRAPP LG,CARAMEL FRAP LG,PLU_4251515,CARAMEL FRAP LG,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-08-29,160,160,856.00,SPELLING_CORRECTION,Standardised FRAPP to FRAP to match the associ...,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1


### I am not changing any of the manual review needed names because it is not necessary.

In [10]:
# ============================================================
# Step 1 - Final Lock Cell
# Close the eight reviewed names without changing them
# ============================================================

from pathlib import Path

import pandas as pd


DATA_FOLDER = Path("eden_datasets")


STEP1_DRAFT_MAPPING_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling.csv"
)


STEP1_LOCKED_MAPPING_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling_locked.csv"
)


step1_locked_mapping = pd.read_csv(
    STEP1_DRAFT_MAPPING_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


reviewed_no_change_codes = {
    4241404: "ODONNELL /POPCORN",
    2024241: "JOS POWERBALL",
    2000000269: "BALLYGOWAN SPARK WATER 330ML",
    4241537: "CRUSTED CROISSANTS",
    4241455: "ICED VANILLA 16OZ",
    4241405: "HUNKY DORY",
    4241409: "PACKS OF SWEET - WINE GUMS",
    4241415: "LUXURY MILLIONAIRES FLAPJACK",
}


for plu_code, expected_name in reviewed_no_change_codes.items():

    row_mask = (
        step1_locked_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(row_mask.sum()) == 1, (
        f"Expected exactly one row for PLUCode {plu_code}."
    )


    actual_name = (
        step1_locked_mapping.loc[
            row_mask,
            "PLUName_Original"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    assert actual_name == expected_name.upper(), (
        f"Unexpected product name for PLUCode {plu_code}.\n"
        f"Expected: {expected_name}\n"
        f"Found: {actual_name}"
    )


    # The product name remains unchanged.
    step1_locked_mapping.loc[
        row_mask,
        "PLUName_Corrected"
    ] = actual_name


    step1_locked_mapping.loc[
        row_mask,
        "CanonicalProductName"
    ] = actual_name


    step1_locked_mapping.loc[
        row_mask,
        "MappingType"
    ] = "REVIEWED_NO_CHANGE"


    step1_locked_mapping.loc[
        row_mask,
        "MappingReason"
    ] = (
        "Name is consistently recorded throughout the dataset "
        "and the suggested alternative cannot be confirmed."
    )


    step1_locked_mapping.loc[
        row_mask,
        "MappingConfidence"
    ] = "REVIEWED"


    step1_locked_mapping.loc[
        row_mask,
        "MappingStatus"
    ] = "APPROVED_STEP_1"


    step1_locked_mapping.loc[
        row_mask,
        "MappingVersion"
    ] = "STEP_1_SPELLING_FINAL"


# ------------------------------------------------------------
# Final Step 1 validation
# ------------------------------------------------------------

assert len(step1_locked_mapping) == 236

assert step1_locked_mapping[
    "PLUCode"
].is_unique

assert step1_locked_mapping[
    "CanonicalProductID"
].is_unique

assert step1_locked_mapping[
    "CanonicalProductID"
].nunique() == 236

assert not step1_locked_mapping[
    "MappingStatus"
].eq("PENDING_MANUAL_REVIEW").any(), (
    "Step 1 still contains pending manual-review products."
)

assert int(
    step1_locked_mapping[
        "MappingType"
    ].eq("SPELLING_CORRECTION").sum()
) == 25

assert int(
    step1_locked_mapping[
        "MappingType"
    ].eq("REVIEWED_NO_CHANGE").sum()
) == 8

assert int(
    step1_locked_mapping["TotalUnits"].sum()
) == 141_480


step1_locked_mapping.to_csv(
    STEP1_LOCKED_MAPPING_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("=" * 70)
print("STEP 1 LOCKED SUCCESSFULLY")
print("=" * 70)
print()
print("Mapping rows:", len(step1_locked_mapping))
print(
    "Spelling corrections:",
    int(
        step1_locked_mapping[
            "MappingType"
        ].eq("SPELLING_CORRECTION").sum()
    )
)
print(
    "Reviewed with no change:",
    int(
        step1_locked_mapping[
            "MappingType"
        ].eq("REVIEWED_NO_CHANGE").sum()
    )
)
print(
    "Pending reviews:",
    int(
        step1_locked_mapping[
            "MappingStatus"
        ].eq("PENDING_MANUAL_REVIEW").sum()
    )
)
print(
    "Canonical product IDs:",
    step1_locked_mapping[
        "CanonicalProductID"
    ].nunique()
)
print()
print("Locked Step 1 mapping:")
print(STEP1_LOCKED_MAPPING_FILE)

STEP 1 LOCKED SUCCESSFULLY

Mapping rows: 236
Spelling corrections: 25
Reviewed with no change: 8
Pending reviews: 0
Canonical product IDs: 236

Locked Step 1 mapping:
eden_datasets/UL_EDEN_master_product_mapping_step1_spelling_locked.csv


## Next is to normalise the branded beverags like there was Bewleys ameriano there was RT americano. I am going to make them all to one but keep the tea 16 oz efor example as it is or change it to standard

In [11]:
# ============================================================
# Step 2 - Cell 0
# Finalise Step 1 by correcting RN CAPPUCINO
#
# This creates a new Step 1 final file.
# It does not overwrite the existing locked mapping.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


STEP1_LOCKED_MAPPING_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling_locked.csv"
)


STEP1_FINAL_MAPPING_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling_final.csv"
)


if not STEP1_LOCKED_MAPPING_FILE.exists():
    raise FileNotFoundError(
        "The locked Step 1 mapping was not found:\n"
        f"{STEP1_LOCKED_MAPPING_FILE}"
    )


step1_final_mapping = pd.read_csv(
    STEP1_LOCKED_MAPPING_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


step1_final_mapping["PLUCode"] = (
    pd.to_numeric(
        step1_final_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step1_final_mapping["TotalUnits"] = (
    pd.to_numeric(
        step1_final_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


# ------------------------------------------------------------
# Validate the locked Step 1 mapping
# ------------------------------------------------------------

assert len(step1_final_mapping) == 236, (
    "Step 1 mapping should contain 236 PLU codes."
)


assert step1_final_mapping["PLUCode"].is_unique, (
    "PLUCode is not unique in the Step 1 mapping."
)


assert step1_final_mapping[
    "CanonicalProductID"
].is_unique, (
    "Step 1 should not contain any product merges."
)


assert not step1_final_mapping[
    "MappingStatus"
].eq("PENDING_MANUAL_REVIEW").any(), (
    "Step 1 still contains pending manual-review products."
)


assert int(
    step1_final_mapping["TotalUnits"].sum()
) == 141_480, (
    "Step 1 mapping UnitSold total is incorrect."
)


# ------------------------------------------------------------
# Correct the one missed definite spelling error
# ------------------------------------------------------------

rn_cappuccino_plu = 44382


rn_cappuccino_mask = (
    step1_final_mapping["PLUCode"]
    .eq(rn_cappuccino_plu)
)


assert int(rn_cappuccino_mask.sum()) == 1, (
    "Expected exactly one mapping row for PLUCode 44382."
)


actual_original_name = (
    step1_final_mapping.loc[
        rn_cappuccino_mask,
        "PLUName_Original"
    ]
    .iloc[0]
    .strip()
    .upper()
)


assert actual_original_name == "RN CAPPUCINO", (
    "Unexpected original name for PLUCode 44382.\n"
    f"Found: {actual_original_name}"
)


current_corrected_name = (
    step1_final_mapping.loc[
        rn_cappuccino_mask,
        "PLUName_Corrected"
    ]
    .iloc[0]
    .strip()
    .upper()
)


assert current_corrected_name in {
    "RN CAPPUCINO",
    "RN CAPPUCCINO",
}, (
    "Unexpected corrected name for PLUCode 44382.\n"
    f"Found: {current_corrected_name}"
)


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "PLUName_Corrected"
] = "RN CAPPUCCINO"


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "CanonicalProductName"
] = "RN CAPPUCCINO"


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "MappingType"
] = "SPELLING_CORRECTION"


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "MappingReason"
] = (
    "Corrected CAPPUCINO to CAPPUCCINO."
)


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "MappingConfidence"
] = "HIGH"


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "MappingStatus"
] = "APPROVED_STEP_1"


# Apply one consistent version to the complete final mapping
step1_final_mapping[
    "MappingVersion"
] = "STEP_1_SPELLING_FINAL_V2"


# ------------------------------------------------------------
# Validate final Step 1 state
# ------------------------------------------------------------

assert int(
    step1_final_mapping[
        "MappingType"
    ].eq("SPELLING_CORRECTION").sum()
) == 26, (
    "The final Step 1 mapping should contain "
    "26 definite spelling corrections."
)


assert int(
    step1_final_mapping[
        "MappingType"
    ].eq("REVIEWED_NO_CHANGE").sum()
) == 8, (
    "Expected eight reviewed names retained unchanged."
)


assert step1_final_mapping[
    "CanonicalProductID"
].nunique() == 236


assert int(
    step1_final_mapping["TotalUnits"].sum()
) == 141_480


step1_final_mapping.to_csv(
    STEP1_FINAL_MAPPING_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("=" * 72)
print("STEP 1 FINAL MAPPING CREATED")
print("=" * 72)
print()
print(
    "Mapping rows:",
    len(step1_final_mapping)
)
print(
    "Spelling corrections:",
    int(
        step1_final_mapping[
            "MappingType"
        ].eq("SPELLING_CORRECTION").sum()
    )
)
print(
    "Reviewed without change:",
    int(
        step1_final_mapping[
            "MappingType"
        ].eq("REVIEWED_NO_CHANGE").sum()
    )
)
print(
    "Pending reviews:",
    int(
        step1_final_mapping[
            "MappingStatus"
        ].eq("PENDING_MANUAL_REVIEW").sum()
    )
)
print()
print("Final Step 1 file:")
print(STEP1_FINAL_MAPPING_FILE)

STEP 1 FINAL MAPPING CREATED

Mapping rows: 236
Spelling corrections: 26
Reviewed without change: 8
Pending reviews: 0

Final Step 1 file:
eden_datasets/UL_EDEN_master_product_mapping_step1_spelling_final.csv


In [12]:
# ============================================================
# Step 2 - Cell 1
# Load final Step 1 mapping and corrected transactions
# ============================================================

TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


STEP2_MAPPING_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step2_branded_beverages.csv"
)


STEP2_PLU_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step2_branded_beverage_plu_audit.csv"
)


STEP2_CANONICAL_SUMMARY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step2_branded_beverage_canonical_summary.csv"
)


STEP2_SUPPLIER_PERIOD_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step2_branded_beverage_supplier_periods.csv"
)


if not TRANSACTION_INPUT_FILE.exists():
    raise FileNotFoundError(
        "Corrected transaction file not found:\n"
        f"{TRANSACTION_INPUT_FILE}"
    )


if not STEP1_FINAL_MAPPING_FILE.exists():
    raise FileNotFoundError(
        "Final Step 1 mapping not found:\n"
        f"{STEP1_FINAL_MAPPING_FILE}"
    )


step2_transactions = pd.read_csv(
    TRANSACTION_INPUT_FILE
)


step2_mapping = pd.read_csv(
    STEP1_FINAL_MAPPING_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


# Preserve exact source-column order for later validation
step2_source_columns = (
    step2_transactions.columns.tolist()
)


# ------------------------------------------------------------
# Convert transaction columns safely
# ------------------------------------------------------------

step2_transactions["TransDate"] = pd.to_datetime(
    step2_transactions["TransDate"],
    errors="raise"
)


step2_transactions["Date"] = (
    pd.to_datetime(
        step2_transactions["Date"],
        errors="raise"
    )
    .dt.normalize()
)


step2_transactions["PLUCode"] = (
    pd.to_numeric(
        step2_transactions["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step2_transactions["UnitSold"] = (
    pd.to_numeric(
        step2_transactions["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


step2_transactions["TransValue"] = pd.to_numeric(
    step2_transactions["TransValue"],
    errors="raise"
)


step2_mapping["PLUCode"] = (
    pd.to_numeric(
        step2_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step2_mapping["TotalUnits"] = (
    pd.to_numeric(
        step2_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


# ------------------------------------------------------------
# Record immutable source totals
# ------------------------------------------------------------

step2_source_row_count = len(
    step2_transactions
)


step2_source_unit_total = int(
    step2_transactions["UnitSold"].sum()
)


step2_source_value_cents = int(
    (
        step2_transactions["TransValue"]
        * 100
    )
    .round()
    .sum()
)


# ------------------------------------------------------------
# Validate inputs
# ------------------------------------------------------------

assert step2_source_row_count == 138_983


assert step2_source_unit_total == 141_480


assert step2_transactions[
    "PLUCode"
].nunique() == 236


assert len(step2_mapping) == 236


assert step2_mapping[
    "PLUCode"
].is_unique


assert step2_mapping[
    "CanonicalProductID"
].is_unique


assert step2_mapping[
    "CanonicalProductID"
].nunique() == 236


assert int(
    step2_mapping["TotalUnits"].sum()
) == 141_480


assert not step2_mapping[
    "MappingStatus"
].eq("PENDING_MANUAL_REVIEW").any()


print("Step 2 source files loaded and validated.")
print()
print(
    "Transaction rows:",
    f"{step2_source_row_count:,}"
)
print(
    "Transaction units:",
    f"{step2_source_unit_total:,}"
)
print(
    "Step 1 PLU identities:",
    step2_mapping[
        "CanonicalProductID"
    ].nunique()
)

Step 2 source files loaded and validated.

Transaction rows: 138,983
Transaction units: 141,480
Step 1 PLU identities: 236


In [13]:
# ============================================================
# Step 2 - Cell 2
# Verify one consistent product identity per PLUCode
# ============================================================

source_plu_consistency = (
    step2_transactions
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        SourceNameCount=(
            "PLUName",
            "nunique"
        ),
        SourceGroupCodeCount=(
            "GroupCode",
            "nunique"
        ),
        SourceGroupNameCount=(
            "GroupName",
            "nunique"
        ),
        SourcePLUName=(
            "PLUName",
            "first"
        ),
        SourceGroupCode=(
            "GroupCode",
            "first"
        ),
        SourceGroupName=(
            "GroupName",
            "first"
        ),
        SourceRows=(
            "PLUCode",
            "size"
        ),
        SourceUnits=(
            "UnitSold",
            "sum"
        ),
    )
)


assert source_plu_consistency[
    "SourceNameCount"
].eq(1).all(), (
    "At least one PLUCode has multiple product names."
)


assert source_plu_consistency[
    "SourceGroupCodeCount"
].eq(1).all(), (
    "At least one PLUCode has multiple GroupCode values."
)


assert source_plu_consistency[
    "SourceGroupNameCount"
].eq(1).all(), (
    "At least one PLUCode has multiple GroupName values."
)


mapping_source_check = step2_mapping[
    [
        "PLUCode",
        "PLUName_Original",
        "GroupCode",
        "GroupName",
        "TotalUnits",
    ]
].merge(
    source_plu_consistency,
    on="PLUCode",
    how="outer",
    validate="one_to_one"
)


assert len(mapping_source_check) == 236


assert mapping_source_check[
    "PLUName_Original"
].notna().all()


assert mapping_source_check[
    "SourcePLUName"
].notna().all()


name_matches = (
    mapping_source_check[
        "PLUName_Original"
    ]
    .astype("string")
    .str.strip()
    ==
    mapping_source_check[
        "SourcePLUName"
    ]
    .astype("string")
    .str.strip()
)


assert name_matches.all(), (
    "A Step 1 original product name does not match "
    "the transaction source."
)


assert (
    mapping_source_check["GroupCode"]
    ==
    mapping_source_check["SourceGroupCode"]
).all()


assert (
    mapping_source_check["GroupName"]
    .astype("string")
    .str.strip()
    ==
    mapping_source_check["SourceGroupName"]
    .astype("string")
    .str.strip()
).all()


assert (
    mapping_source_check["TotalUnits"]
    ==
    mapping_source_check["SourceUnits"]
).all(), (
    "A PLU-level unit total differs between the mapping "
    "and the source transaction dataset."
)


print("Step 1 mapping matches the transaction source exactly.")

Step 1 mapping matches the transaction source exactly.


In [14]:
# ============================================================
# Step 2 - Cell 3
# Create isolated Step 2 product-identity columns
# ============================================================

# Preserve the final Step 1 identity explicitly
step2_mapping[
    "CanonicalProductID_Step1"
] = step2_mapping[
    "CanonicalProductID"
].astype("string")


step2_mapping[
    "CanonicalProductName_Step1"
] = step2_mapping[
    "CanonicalProductName"
].astype("string")


# Step 2 initially equals Step 1 for every product
step2_mapping[
    "CanonicalProductID_Step2"
] = step2_mapping[
    "CanonicalProductID_Step1"
].copy()


step2_mapping[
    "CanonicalProductName_Step2"
] = step2_mapping[
    "CanonicalProductName_Step1"
].copy()


# New Step 2 metadata
step2_mapping[
    "BeverageSeries_Step2"
] = pd.Series(
    pd.NA,
    index=step2_mapping.index,
    dtype="string"
)


step2_mapping[
    "BeverageType_Step2"
] = pd.Series(
    pd.NA,
    index=step2_mapping.index,
    dtype="string"
)


step2_mapping[
    "SupplierLabel_Step2"
] = pd.Series(
    pd.NA,
    index=step2_mapping.index,
    dtype="string"
)


step2_mapping[
    "ProductFamily_Step2"
] = pd.Series(
    pd.NA,
    index=step2_mapping.index,
    dtype="string"
)


step2_mapping[
    "SupplierNormalizationApplied_Step2"
] = False


step2_mapping[
    "SupplierNormalizationType_Step2"
] = "NOT_APPLICABLE"


step2_mapping[
    "SupplierNormalizationReason_Step2"
] = (
    "Product identity retained from Step 1."
)


step2_mapping[
    "SupplierNormalizationStatus_Step2"
] = "UNCHANGED_STEP_2"


step2_mapping[
    "SupplierNormalizationVersion_Step2"
] = (
    "STEP_2_BRANDED_BEVERAGES_V1"
)


print("Isolated Step 2 columns created.")

Isolated Step 2 columns created.


In [15]:
# ============================================================
# Step 2 - Cell 4
# Define confirmed branded-beverage mappings
# ============================================================

branded_beverage_mappings = [
    # --------------------------------------------------------
    # B-AMERICANO
    # --------------------------------------------------------
    {
        "PLUCode": 42527101,
        "ExpectedCorrectedName": "BEWLEYS AMERICANO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_AMERICANO",
        "CanonicalProductName_Step2":
            "B-AMERICANO",
        "BeverageType_Step2": "AMERICANO",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44381,
        "ExpectedCorrectedName": "RN AMERICANO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_AMERICANO",
        "CanonicalProductName_Step2":
            "B-AMERICANO",
        "BeverageType_Step2": "AMERICANO",
        "SupplierLabel_Step2": "RN",
    },
    {
        "PLUCode": 425531,
        "ExpectedCorrectedName":
            "ROASTED NOTES AMERICANO CT",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_AMERICANO",
        "CanonicalProductName_Step2":
            "B-AMERICANO",
        "BeverageType_Step2": "AMERICANO",
        "SupplierLabel_Step2": "CT",
    },

    # --------------------------------------------------------
    # B-CAPPUCCINO
    # --------------------------------------------------------
    {
        "PLUCode": 42527103,
        "ExpectedCorrectedName":
            "BEWLEYS CAPPUCCINO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_CAPPUCCINO",
        "CanonicalProductName_Step2":
            "B-CAPPUCCINO",
        "BeverageType_Step2": "CAPPUCCINO",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44382,
        "ExpectedCorrectedName":
            "RN CAPPUCCINO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_CAPPUCCINO",
        "CanonicalProductName_Step2":
            "B-CAPPUCCINO",
        "BeverageType_Step2": "CAPPUCCINO",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-FLAT WHITE
    # --------------------------------------------------------
    {
        "PLUCode": 42527104,
        "ExpectedCorrectedName":
            "BEWLEYS FLAT WHITE",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_FLAT_WHITE",
        "CanonicalProductName_Step2":
            "B-FLAT WHITE",
        "BeverageType_Step2": "FLAT WHITE",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 417173,
        "ExpectedCorrectedName":
            "RN FLAT WHITE",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_FLAT_WHITE",
        "CanonicalProductName_Step2":
            "B-FLAT WHITE",
        "BeverageType_Step2": "FLAT WHITE",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-LATTE
    # --------------------------------------------------------
    {
        "PLUCode": 42527102,
        "ExpectedCorrectedName":
            "BEWLEYS LATTE",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_LATTE",
        "CanonicalProductName_Step2":
            "B-LATTE",
        "BeverageType_Step2": "LATTE",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44384,
        "ExpectedCorrectedName":
            "RN LATTE",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_LATTE",
        "CanonicalProductName_Step2":
            "B-LATTE",
        "BeverageType_Step2": "LATTE",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-MOCHA
    # --------------------------------------------------------
    {
        "PLUCode": 42527105,
        "ExpectedCorrectedName":
            "BEWLEYS MOCHA",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_MOCHA",
        "CanonicalProductName_Step2":
            "B-MOCHA",
        "BeverageType_Step2": "MOCHA",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44385,
        "ExpectedCorrectedName":
            "RN MOCHA",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_MOCHA",
        "CanonicalProductName_Step2":
            "B-MOCHA",
        "BeverageType_Step2": "MOCHA",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-HOT CHOCOLATE
    # --------------------------------------------------------
    {
        "PLUCode": 42527106,
        "ExpectedCorrectedName":
            "BEWLEYS HOT CHOC",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_HOT_CHOCOLATE",
        "CanonicalProductName_Step2":
            "B-HOT CHOCOLATE",
        "BeverageType_Step2": "HOT CHOCOLATE",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44386,
        "ExpectedCorrectedName":
            "RN HOT CHOC",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_HOT_CHOCOLATE",
        "CanonicalProductName_Step2":
            "B-HOT CHOCOLATE",
        "BeverageType_Step2": "HOT CHOCOLATE",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-ESPRESSO
    # --------------------------------------------------------
    {
        "PLUCode": 4241523,
        "ExpectedCorrectedName":
            "ROASTED NOTES ESPRESSO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_ESPRESSO",
        "CanonicalProductName_Step2":
            "B-ESPRESSO",
        "BeverageType_Step2": "ESPRESSO",
        "SupplierLabel_Step2": "RN",
    },
    {
        "PLUCode": 42555,
        "ExpectedCorrectedName":
            "ROASTED NOTES ESPRESSO CT",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_ESPRESSO",
        "CanonicalProductName_Step2":
            "B-ESPRESSO",
        "BeverageType_Step2": "ESPRESSO",
        "SupplierLabel_Step2": "CT",
    },

    # --------------------------------------------------------
    # B-FILTER COFFEE
    # --------------------------------------------------------
    {
        "PLUCode": 42563,
        "ExpectedCorrectedName":
            "FILTER COFFEE CT",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_FILTER_COFFEE",
        "CanonicalProductName_Step2":
            "B-FILTER COFFEE",
        "BeverageType_Step2": "FILTER COFFEE",
        "SupplierLabel_Step2": "CT",
    },

    # --------------------------------------------------------
    # B-TEA
    # --------------------------------------------------------
    {
        "PLUCode": 42561,
        "ExpectedCorrectedName":
            "TEA CT",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_TEA",
        "CanonicalProductName_Step2":
            "B-TEA",
        "BeverageType_Step2": "TEA",
        "SupplierLabel_Step2": "CT",
    },
]


branded_beverage_mapping_df = pd.DataFrame(
    branded_beverage_mappings
)


assert len(branded_beverage_mapping_df) == 17


assert branded_beverage_mapping_df[
    "PLUCode"
].is_unique


assert branded_beverage_mapping_df[
    "CanonicalProductID_Step2"
].nunique() == 9


assert branded_beverage_mapping_df[
    "CanonicalProductName_Step2"
].nunique() == 9


approved_branded_plu_codes = set(
    branded_beverage_mapping_df["PLUCode"]
)


print("Confirmed branded-beverage mapping created.")
print()
print(
    "Source PLUs:",
    len(branded_beverage_mapping_df)
)
print(
    "Canonical branded beverages:",
    branded_beverage_mapping_df[
        "CanonicalProductID_Step2"
    ].nunique()
)

display(
    branded_beverage_mapping_df.sort_values(
        [
            "CanonicalProductName_Step2",
            "SupplierLabel_Step2",
        ]
    )
)

Confirmed branded-beverage mapping created.

Source PLUs: 17
Canonical branded beverages: 9


,PLUCode,ExpectedCorrectedName,CanonicalProductID_Step2,CanonicalProductName_Step2,BeverageType_Step2,SupplierLabel_Step2
0,42527101,BEWLEYS AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,AMERICANO,BEWLEYS
2,425531,ROASTED NOTES AMERICANO CT,BEV_BRANDED_AMERICANO,B-AMERICANO,AMERICANO,CT
1,44381,RN AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,AMERICANO,RN
3,42527103,BEWLEYS CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,CAPPUCCINO,BEWLEYS
4,44382,RN CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,CAPPUCCINO,RN
14,42555,ROASTED NOTES ESPRESSO CT,BEV_BRANDED_ESPRESSO,B-ESPRESSO,ESPRESSO,CT
13,4241523,ROASTED NOTES ESPRESSO,BEV_BRANDED_ESPRESSO,B-ESPRESSO,ESPRESSO,RN
15,42563,FILTER COFFEE CT,BEV_BRANDED_FILTER_COFFEE,B-FILTER COFFEE,FILTER COFFEE,CT
5,42527104,BEWLEYS FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,FLAT WHITE,BEWLEYS
6,417173,RN FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,FLAT WHITE,RN


In [16]:
# ============================================================
# Step 2 - Cell 5
# Validate every proposed mapping before applying it
# ============================================================

source_branded_transactions = (
    step2_transactions.loc[
        step2_transactions[
            "PLUCode"
        ].isin(approved_branded_plu_codes)
    ]
    .copy()
)


assert source_branded_transactions[
    "PLUCode"
].nunique() == 17


assert len(source_branded_transactions) == 12_165


assert int(
    source_branded_transactions["UnitSold"].sum()
) == 12_165


assert source_branded_transactions[
    "GroupName"
].eq("HOT BEVS").all(), (
    "At least one approved branded PLU is not in HOT BEVS."
)


for proposed_mapping in branded_beverage_mappings:

    plu_code = proposed_mapping["PLUCode"]

    mapping_row_mask = (
        step2_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(mapping_row_mask.sum()) == 1, (
        f"Expected one Step 1 mapping row for PLU {plu_code}."
    )


    actual_corrected_name = (
        step2_mapping.loc[
            mapping_row_mask,
            "PLUName_Corrected"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    expected_corrected_name = (
        proposed_mapping[
            "ExpectedCorrectedName"
        ]
        .strip()
        .upper()
    )


    assert actual_corrected_name == (
        expected_corrected_name
    ), (
        f"Corrected-name mismatch for PLU {plu_code}.\n"
        f"Expected: {expected_corrected_name}\n"
        f"Found: {actual_corrected_name}"
    )


    source_product_rows = (
        step2_transactions.loc[
            step2_transactions[
                "PLUCode"
            ].eq(plu_code)
        ]
    )


    assert len(source_product_rows) > 0


    assert source_product_rows[
        "PLUName"
    ].nunique() == 1


    assert source_product_rows[
        "GroupName"
    ].eq("HOT BEVS").all()


print("All 17 proposed branded-beverage mappings passed preflight.")
print()
print(
    "Transaction rows covered:",
    f"{len(source_branded_transactions):,}"
)
print(
    "Units covered:",
    f"{int(source_branded_transactions['UnitSold'].sum()):,}"
)

All 17 proposed branded-beverage mappings passed preflight.

Transaction rows covered: 12,165
Units covered: 12,165


In [17]:
# ============================================================
# Step 2 - Cell 6
# Apply mappings only to the 17 confirmed branded PLUs
# ============================================================

for approved_mapping in branded_beverage_mappings:

    plu_code = approved_mapping["PLUCode"]

    row_mask = (
        step2_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(row_mask.sum()) == 1


    step2_mapping.loc[
        row_mask,
        "CanonicalProductID_Step2"
    ] = approved_mapping[
        "CanonicalProductID_Step2"
    ]


    step2_mapping.loc[
        row_mask,
        "CanonicalProductName_Step2"
    ] = approved_mapping[
        "CanonicalProductName_Step2"
    ]


    step2_mapping.loc[
        row_mask,
        "BeverageSeries_Step2"
    ] = "BRANDED"


    step2_mapping.loc[
        row_mask,
        "BeverageType_Step2"
    ] = approved_mapping[
        "BeverageType_Step2"
    ]


    step2_mapping.loc[
        row_mask,
        "SupplierLabel_Step2"
    ] = approved_mapping[
        "SupplierLabel_Step2"
    ]


    step2_mapping.loc[
        row_mask,
        "ProductFamily_Step2"
    ] = "HOT BEVERAGE"


    step2_mapping.loc[
        row_mask,
        "SupplierNormalizationApplied_Step2"
    ] = True


    step2_mapping.loc[
        row_mask,
        "SupplierNormalizationType_Step2"
    ] = "CONFIRMED_BRANDED_EQUIVALENCE"


    step2_mapping.loc[
        row_mask,
        "SupplierNormalizationReason_Step2"
    ] = (
        "Confirmed branded beverage equivalent sold "
        "under Bewleys, RN or CT supplier labels."
    )


    step2_mapping.loc[
        row_mask,
        "SupplierNormalizationStatus_Step2"
    ] = "APPROVED_STEP_2"


print(
    "Branded PLU mappings applied:",
    int(
        step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ].sum()
    )
)

Branded PLU mappings applied: 17


In [18]:
# ============================================================
# Step 2 - Cell 7
# Validate isolation and prevent accidental product merging
# ============================================================

step2_changed_identity_mask = (
    step2_mapping[
        "CanonicalProductID_Step2"
    ]
    !=
    step2_mapping[
        "CanonicalProductID_Step1"
    ]
)


changed_plu_codes = set(
    step2_mapping.loc[
        step2_changed_identity_mask,
        "PLUCode",
    ]
)


assert changed_plu_codes == approved_branded_plu_codes, (
    "A non-approved product was changed in Step 2."
)


assert int(
    step2_changed_identity_mask.sum()
) == 17


# ------------------------------------------------------------
# Every other product must remain exactly as Step 1
# ------------------------------------------------------------

unchanged_step2_mask = (
    ~step2_mapping[
        "PLUCode"
    ].isin(approved_branded_plu_codes)
)


assert (
    step2_mapping.loc[
        unchanged_step2_mask,
        "CanonicalProductID_Step2"
    ]
    .reset_index(drop=True)
    .equals(
        step2_mapping.loc[
            unchanged_step2_mask,
            "CanonicalProductID_Step1"
        ]
        .reset_index(drop=True)
    )
), (
    "A non-branded product canonical ID changed."
)


assert (
    step2_mapping.loc[
        unchanged_step2_mask,
        "CanonicalProductName_Step2"
    ]
    .reset_index(drop=True)
    .equals(
        step2_mapping.loc[
            unchanged_step2_mask,
            "CanonicalProductName_Step1"
        ]
        .reset_index(drop=True)
    )
), (
    "A non-branded product canonical name changed."
)


# ------------------------------------------------------------
# Standard HOT BEVS must remain unchanged
# ------------------------------------------------------------

standard_hot_beverage_mask = (
    step2_mapping["GroupName"].eq("HOT BEVS")
    &
    ~step2_mapping[
        "PLUCode"
    ].isin(approved_branded_plu_codes)
)


assert (
    step2_mapping.loc[
        standard_hot_beverage_mask,
        "CanonicalProductID_Step2"
    ]
    .reset_index(drop=True)
    .equals(
        step2_mapping.loc[
            standard_hot_beverage_mask,
            "CanonicalProductID_Step1"
        ]
        .reset_index(drop=True)
    )
), (
    "A standard restaurant beverage was accidentally merged."
)


# ------------------------------------------------------------
# Expected identity counts
# ------------------------------------------------------------

assert step2_mapping[
    "CanonicalProductID_Step1"
].nunique() == 236


assert step2_mapping[
    "CanonicalProductID_Step2"
].nunique() == 228, (
    "Expected 228 canonical identities after merging "
    "17 source PLUs into nine branded beverages."
)


assert step2_mapping.loc[
    step2_changed_identity_mask,
    "CanonicalProductName_Step2"
].str.startswith("B-").all()


assert step2_mapping.loc[
    step2_changed_identity_mask,
    "BeverageSeries_Step2"
].eq("BRANDED").all()


# Confirm KIMBOX and KIMBOCK were untouched
kim_product_mask = (
    step2_mapping["PLUName_Corrected"]
    .astype("string")
    .str.contains(
        r"KIMBOX|KIMBOCK",
        case=False,
        na=False,
        regex=True
    )
)


assert (
    step2_mapping.loc[
        kim_product_mask,
        "CanonicalProductID_Step2"
    ]
    .reset_index(drop=True)
    .equals(
        step2_mapping.loc[
            kim_product_mask,
            "CanonicalProductID_Step1"
        ]
        .reset_index(drop=True)
    )
)


print("Step 2 isolation checks passed.")
print()
print(
    "Step 1 canonical identities:",
    step2_mapping[
        "CanonicalProductID_Step1"
    ].nunique()
)
print(
    "Step 2 canonical identities:",
    step2_mapping[
        "CanonicalProductID_Step2"
    ].nunique()
)
print(
    "PLUs changed:",
    int(step2_changed_identity_mask.sum())
)
print(
    "Standard HOT BEVS changed:",
    int(
        (
            standard_hot_beverage_mask
            &
            step2_changed_identity_mask
        ).sum()
    )
)

Step 2 isolation checks passed.

Step 1 canonical identities: 236
Step 2 canonical identities: 228
PLUs changed: 17
Standard HOT BEVS changed: 0


In [19]:
# ============================================================
# Step 2 - Cell 8
# Create supplier-period and canonical-beverage audits
# ============================================================

branded_transaction_audit = (
    source_branded_transactions
    .merge(
        branded_beverage_mapping_df[
            [
                "PLUCode",
                "CanonicalProductID_Step2",
                "CanonicalProductName_Step2",
                "BeverageType_Step2",
                "SupplierLabel_Step2",
            ]
        ],
        on="PLUCode",
        how="left",
        validate="many_to_one"
    )
)


supplier_period_summary = (
    branded_transaction_audit
    .groupby(
        "SupplierLabel_Step2",
        as_index=False
    )
    .agg(
        FirstObservedDate=(
            "Date",
            "min"
        ),
        LastObservedDate=(
            "Date",
            "max"
        ),
        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
    )
    .sort_values(
        "FirstObservedDate"
    )
    .reset_index(drop=True)
)


canonical_branded_summary = (
    branded_transaction_audit
    .groupby(
        [
            "CanonicalProductID_Step2",
            "CanonicalProductName_Step2",
            "BeverageType_Step2",
        ],
        as_index=False
    )
    .agg(
        FirstObservedDate=(
            "Date",
            "min"
        ),
        LastObservedDate=(
            "Date",
            "max"
        ),
        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
    )
    .sort_values(
        "TotalUnits",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate observed supplier periods
# ------------------------------------------------------------

bewleys_row = supplier_period_summary.loc[
    supplier_period_summary[
        "SupplierLabel_Step2"
    ].eq("BEWLEYS")
]


rn_row = supplier_period_summary.loc[
    supplier_period_summary[
        "SupplierLabel_Step2"
    ].eq("RN")
]


ct_row = supplier_period_summary.loc[
    supplier_period_summary[
        "SupplierLabel_Step2"
    ].eq("CT")
]


assert len(bewleys_row) == 1
assert len(rn_row) == 1
assert len(ct_row) == 1


assert (
    bewleys_row["LastObservedDate"].iloc[0]
    == pd.Timestamp("2025-10-13")
)


assert (
    rn_row["FirstObservedDate"].iloc[0]
    == pd.Timestamp("2025-10-14")
)


assert (
    ct_row["FirstObservedDate"].iloc[0]
    == pd.Timestamp("2025-10-28")
)


assert (
    ct_row["LastObservedDate"].iloc[0]
    == pd.Timestamp("2025-10-28")
)


assert len(canonical_branded_summary) == 9


assert int(
    canonical_branded_summary["TotalUnits"].sum()
) == 12_165


print("Branded-beverage period audit completed.")
print()
print("Supplier periods:")

display(supplier_period_summary)

print()
print("Canonical branded-beverage summary:")

display(canonical_branded_summary)

Branded-beverage period audit completed.

Supplier periods:


,SupplierLabel_Step2,FirstObservedDate,LastObservedDate,SourcePLUCount,TransactionRows,TotalUnits
0,BEWLEYS,2025-04-01,2025-10-13,6,3261,3261
1,RN,2025-10-14,2026-03-30,7,8900,8900
2,CT,2025-10-28,2025-10-28,4,4,4



Canonical branded-beverage summary:


,CanonicalProductID_Step2,CanonicalProductName_Step2,BeverageType_Step2,FirstObservedDate,LastObservedDate,SourcePLUCount,TransactionRows,TotalUnits
0,BEV_BRANDED_AMERICANO,B-AMERICANO,AMERICANO,2025-04-01,2026-03-30,3,4812,4812
1,BEV_BRANDED_LATTE,B-LATTE,LATTE,2025-04-01,2026-03-30,2,2739,2739
2,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,CAPPUCCINO,2025-04-01,2026-03-30,2,2284,2284
3,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,FLAT WHITE,2025-04-01,2026-03-30,2,1350,1350
4,BEV_BRANDED_HOT_CHOCOLATE,B-HOT CHOCOLATE,HOT CHOCOLATE,2025-04-02,2026-03-30,2,484,484
5,BEV_BRANDED_MOCHA,B-MOCHA,MOCHA,2025-04-01,2026-03-30,2,454,454
6,BEV_BRANDED_ESPRESSO,B-ESPRESSO,ESPRESSO,2025-10-17,2026-03-30,2,40,40
7,BEV_BRANDED_FILTER_COFFEE,B-FILTER COFFEE,FILTER COFFEE,2025-10-28,2025-10-28,1,1,1
8,BEV_BRANDED_TEA,B-TEA,TEA,2025-10-28,2025-10-28,1,1,1


In [20]:
# ============================================================
# Step 2 - Cell 9
# Join for validation only and prove no source data changed
# ============================================================

step2_transactions_with_order = (
    step2_transactions.copy()
)


step2_transactions_with_order[
    "_SourceRowOrder"
] = np.arange(
    len(step2_transactions_with_order)
)


step2_validation_join = (
    step2_transactions_with_order
    .merge(
        step2_mapping[
            [
                "PLUCode",
                "PLUName_Original",
                "PLUName_Corrected",
                "CanonicalProductID_Step1",
                "CanonicalProductName_Step1",
                "CanonicalProductID_Step2",
                "CanonicalProductName_Step2",
                "BeverageSeries_Step2",
                "BeverageType_Step2",
                "SupplierLabel_Step2",
                "SupplierNormalizationApplied_Step2",
            ]
        ],
        on="PLUCode",
        how="left",
        validate="many_to_one",
        sort=False
    )
    .sort_values("_SourceRowOrder")
    .reset_index(drop=True)
)


source_for_comparison = (
    step2_transactions
    .reset_index(drop=True)
)


assert len(step2_validation_join) == (
    step2_source_row_count
)


assert step2_validation_join[
    "CanonicalProductID_Step2"
].notna().all()


# ------------------------------------------------------------
# Confirm every original source column remained unchanged
# ------------------------------------------------------------

for source_column in step2_source_columns:

    assert (
        step2_validation_join[source_column]
        .reset_index(drop=True)
        .equals(
            source_for_comparison[source_column]
            .reset_index(drop=True)
        )
    ), (
        f"Source column changed during validation join: "
        f"{source_column}"
    )


assert int(
    step2_validation_join["UnitSold"].sum()
) == step2_source_unit_total


joined_value_cents = int(
    (
        step2_validation_join["TransValue"]
        * 100
    )
    .round()
    .sum()
)


assert joined_value_cents == step2_source_value_cents


mapped_transaction_mask = (
    step2_validation_join[
        "SupplierNormalizationApplied_Step2"
    ]
)


assert int(
    mapped_transaction_mask.sum()
) == 12_165


assert int(
    step2_validation_join.loc[
        mapped_transaction_mask,
        "UnitSold",
    ].sum()
) == 12_165


# OPEN UL must never be affected
open_ul_validation_mask = (
    step2_validation_join["PLUName"]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


assert not step2_validation_join.loc[
    open_ul_validation_mask,
    "SupplierNormalizationApplied_Step2"
].any()


print("Full transaction validation passed.")
print()
print(
    "Source rows unchanged:",
    f"{len(step2_validation_join):,}"
)
print(
    "Source units unchanged:",
    f"{int(step2_validation_join['UnitSold'].sum()):,}"
)
print(
    "Source value unchanged:",
    joined_value_cents == step2_source_value_cents
)
print(
    "Branded transaction rows:",
    f"{int(mapped_transaction_mask.sum()):,}"
)
print(
    "Branded units:",
    f"{int(step2_validation_join.loc[mapped_transaction_mask, 'UnitSold'].sum()):,}"
)

Full transaction validation passed.

Source rows unchanged: 138,983
Source units unchanged: 141,480
Source value unchanged: True
Branded transaction rows: 12,165
Branded units: 12,165


In [21]:
# ============================================================
# Step 2 - Cell 10
# Create the detailed branded-beverage PLU audit
# ============================================================

branded_beverage_plu_audit = (
    step2_mapping.loc[
        step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ],
        [
            "PLUCode",
            "PLUName_Original",
            "PLUName_Corrected",
            "CanonicalProductID_Step1",
            "CanonicalProductName_Step1",
            "CanonicalProductID_Step2",
            "CanonicalProductName_Step2",
            "BeverageSeries_Step2",
            "BeverageType_Step2",
            "SupplierLabel_Step2",
            "ProductFamily_Step2",
            "GroupCode",
            "GroupName",
            "FirstObservedDate",
            "LastObservedDate",
            "TransactionRows",
            "TotalUnits",
            "TotalTransactionValue",
            "SupplierNormalizationType_Step2",
            "SupplierNormalizationReason_Step2",
            "SupplierNormalizationStatus_Step2",
            "SupplierNormalizationVersion_Step2",
        ],
    ]
    .sort_values(
        [
            "CanonicalProductName_Step2",
            "FirstObservedDate",
            "SupplierLabel_Step2",
            "PLUCode",
        ]
    )
    .reset_index(drop=True)
)


assert len(branded_beverage_plu_audit) == 17


assert branded_beverage_plu_audit[
    "CanonicalProductID_Step2"
].nunique() == 9


assert int(
    branded_beverage_plu_audit[
        "TotalUnits"
    ].sum()
) == 12_165


display(branded_beverage_plu_audit)

,PLUCode,PLUName_Original,PLUName_Corrected,CanonicalProductID_Step1,CanonicalProductName_Step1,CanonicalProductID_Step2,CanonicalProductName_Step2,BeverageSeries_Step2,BeverageType_Step2,SupplierLabel_Step2,...,GroupName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,TotalTransactionValue,SupplierNormalizationType_Step2,SupplierNormalizationReason_Step2,SupplierNormalizationStatus_Step2,SupplierNormalizationVersion_Step2
0,42527101,BEWLEYS AMERICANO,BEWLEYS AMERICANO,PLU_42527101,BEWLEYS AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,BRANDED,AMERICANO,BEWLEYS,...,HOT BEVS,2025-04-01,2025-10-13,1876,1876,5333.20,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
1,44381,RN AMERICANO,RN AMERICANO,PLU_44381,RN AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,BRANDED,AMERICANO,RN,...,HOT BEVS,2025-10-14,2026-03-30,2935,2935,9392.00,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
2,425531,ROASTED NOTES AMERICANO CT,ROASTED NOTES AMERICANO CT,PLU_425531,ROASTED NOTES AMERICANO CT,BEV_BRANDED_AMERICANO,B-AMERICANO,BRANDED,AMERICANO,CT,...,HOT BEVS,2025-10-28,2025-10-28,1,1,3.50,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
3,42527103,BEWLEYS CAPPUCINO,BEWLEYS CAPPUCCINO,PLU_42527103,BEWLEYS CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,BRANDED,CAPPUCCINO,BEWLEYS,...,HOT BEVS,2025-04-01,2025-10-13,517,517,1614.80,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
4,44382,RN CAPPUCINO,RN CAPPUCCINO,PLU_44382,RN CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,BRANDED,CAPPUCCINO,RN,...,HOT BEVS,2025-10-14,2026-03-30,1767,1767,6361.20,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
5,4241523,ROASTED NOTES ESPRESSO,ROASTED NOTES ESPRESSO,PLU_4241523,ROASTED NOTES ESPRESSO,BEV_BRANDED_ESPRESSO,B-ESPRESSO,BRANDED,ESPRESSO,RN,...,HOT BEVS,2025-10-17,2026-03-30,39,39,124.80,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
6,42555,R0OASTED NOTES EXPRESSO CT,ROASTED NOTES ESPRESSO CT,PLU_42555,ROASTED NOTES ESPRESSO CT,BEV_BRANDED_ESPRESSO,B-ESPRESSO,BRANDED,ESPRESSO,CT,...,HOT BEVS,2025-10-28,2025-10-28,1,1,3.50,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
7,42563,FILTER COFFEE CT,FILTER COFFEE CT,PLU_42563,FILTER COFFEE CT,BEV_BRANDED_FILTER_COFFEE,B-FILTER COFFEE,BRANDED,FILTER COFFEE,CT,...,HOT BEVS,2025-10-28,2025-10-28,1,1,2.50,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
8,42527104,BEWLEYS FLAT WHITE,BEWLEYS FLAT WHITE,PLU_42527104,BEWLEYS FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,BRANDED,FLAT WHITE,BEWLEYS,...,HOT BEVS,2025-04-01,2025-10-13,191,191,577.40,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
9,417173,RN FLAT WHITE,RN FLAT WHITE,PLU_417173,RN FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,BRANDED,FLAT WHITE,RN,...,HOT BEVS,2025-10-14,2026-03-30,1159,1159,4172.40,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1


In [22]:
# ============================================================
# Step 2 - Cell 11
# Save Step 2 mapping and audit outputs
# ============================================================

step2_mapping.to_csv(
    STEP2_MAPPING_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


branded_beverage_plu_audit.to_csv(
    STEP2_PLU_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


canonical_branded_summary.to_csv(
    STEP2_CANONICAL_SUMMARY_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


supplier_period_summary.to_csv(
    STEP2_SUPPLIER_PERIOD_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("Step 2 files saved successfully.")
print()

print("1. Step 2 master mapping:")
print(STEP2_MAPPING_OUTPUT_FILE)

print()
print("2. Branded-beverage PLU audit:")
print(STEP2_PLU_AUDIT_OUTPUT_FILE)

print()
print("3. Canonical branded-beverage summary:")
print(STEP2_CANONICAL_SUMMARY_OUTPUT_FILE)

print()
print("4. Supplier-period audit:")
print(STEP2_SUPPLIER_PERIOD_OUTPUT_FILE)

Step 2 files saved successfully.

1. Step 2 master mapping:
eden_datasets/UL_EDEN_master_product_mapping_step2_branded_beverages.csv

2. Branded-beverage PLU audit:
eden_datasets/UL_EDEN_step2_branded_beverage_plu_audit.csv

3. Canonical branded-beverage summary:
eden_datasets/UL_EDEN_step2_branded_beverage_canonical_summary.csv

4. Supplier-period audit:
eden_datasets/UL_EDEN_step2_branded_beverage_supplier_periods.csv


In [23]:
# ============================================================
# Step 2 - Cell 12
# Reload and validate the saved Step 2 mapping
# ============================================================

saved_step2_mapping = pd.read_csv(
    STEP2_MAPPING_OUTPUT_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


saved_step2_mapping["PLUCode"] = (
    pd.to_numeric(
        saved_step2_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


saved_step2_mapping["TotalUnits"] = (
    pd.to_numeric(
        saved_step2_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


saved_step2_applied = (
    saved_step2_mapping[
        "SupplierNormalizationApplied_Step2"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
    })
)


assert saved_step2_applied.notna().all()


saved_step2_mapping[
    "SupplierNormalizationApplied_Step2"
] = saved_step2_applied.astype(bool)


# ------------------------------------------------------------
# Final saved-file validation
# ------------------------------------------------------------

assert len(saved_step2_mapping) == 236


assert saved_step2_mapping[
    "PLUCode"
].is_unique


assert saved_step2_mapping[
    "CanonicalProductID_Step1"
].nunique() == 236


assert saved_step2_mapping[
    "CanonicalProductID_Step2"
].nunique() == 228


assert int(
    saved_step2_mapping[
        "SupplierNormalizationApplied_Step2"
    ].sum()
) == 17


assert saved_step2_mapping.loc[
    saved_step2_mapping[
        "SupplierNormalizationApplied_Step2"
    ],
    "CanonicalProductID_Step2"
].nunique() == 9


assert saved_step2_mapping.loc[
    saved_step2_mapping[
        "SupplierNormalizationApplied_Step2"
    ],
    "CanonicalProductName_Step2"
].str.startswith("B-").all()


assert int(
    saved_step2_mapping["TotalUnits"].sum()
) == 141_480


saved_branded_unit_total = int(
    saved_step2_mapping.loc[
        saved_step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ],
        "TotalUnits",
    ].sum()
)


assert saved_branded_unit_total == 12_165


# Confirm no non-approved PLU changed identity
saved_changed_mask = (
    saved_step2_mapping[
        "CanonicalProductID_Step2"
    ]
    !=
    saved_step2_mapping[
        "CanonicalProductID_Step1"
    ]
)


assert set(
    saved_step2_mapping.loc[
        saved_changed_mask,
        "PLUCode",
    ]
) == approved_branded_plu_codes


print("=" * 74)
print("STEP 2 BRANDED-BEVERAGE NORMALISATION COMPLETED")
print("=" * 74)
print()
print(
    "Mapping rows:",
    len(saved_step2_mapping)
)
print(
    "Step 1 product identities:",
    saved_step2_mapping[
        "CanonicalProductID_Step1"
    ].nunique()
)
print(
    "Step 2 product identities:",
    saved_step2_mapping[
        "CanonicalProductID_Step2"
    ].nunique()
)
print(
    "Branded source PLUs mapped:",
    int(
        saved_step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ].sum()
    )
)
print(
    "Canonical branded beverages:",
    saved_step2_mapping.loc[
        saved_step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ],
        "CanonicalProductID_Step2",
    ].nunique()
)
print(
    "Branded beverage units:",
    f"{saved_branded_unit_total:,}"
)
print(
    "Source transaction units preserved:",
    f"{step2_source_unit_total:,}"
)
print()
print("Final Step 2 mapping:")
print(STEP2_MAPPING_OUTPUT_FILE)

display(
    saved_step2_mapping.loc[
        saved_step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ],
        [
            "PLUCode",
            "PLUName_Corrected",
            "CanonicalProductID_Step2",
            "CanonicalProductName_Step2",
            "SupplierLabel_Step2",
            "TotalUnits",
        ],
    ]
    .sort_values(
        [
            "CanonicalProductName_Step2",
            "SupplierLabel_Step2",
        ]
    )
)

STEP 2 BRANDED-BEVERAGE NORMALISATION COMPLETED

Mapping rows: 236
Step 1 product identities: 236
Step 2 product identities: 228
Branded source PLUs mapped: 17
Canonical branded beverages: 9
Branded beverage units: 12,165
Source transaction units preserved: 141,480

Final Step 2 mapping:
eden_datasets/UL_EDEN_master_product_mapping_step2_branded_beverages.csv


,PLUCode,PLUName_Corrected,CanonicalProductID_Step2,CanonicalProductName_Step2,SupplierLabel_Step2,TotalUnits
16,42527101,BEWLEYS AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,BEWLEYS,1876
187,425531,ROASTED NOTES AMERICANO CT,BEV_BRANDED_AMERICANO,B-AMERICANO,CT,1
181,44381,RN AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,RN,2935
17,42527103,BEWLEYS CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,BEWLEYS,517
182,44382,RN CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,RN,1767
189,42555,ROASTED NOTES ESPRESSO CT,BEV_BRANDED_ESPRESSO,B-ESPRESSO,CT,1
188,4241523,ROASTED NOTES ESPRESSO,BEV_BRANDED_ESPRESSO,B-ESPRESSO,RN,39
84,42563,FILTER COFFEE CT,BEV_BRANDED_FILTER_COFFEE,B-FILTER COFFEE,CT,1
18,42527104,BEWLEYS FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,BEWLEYS,191
183,417173,RN FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,RN,1159


### Now what I am going to do is forecasting identity of price-tier products independent of later price increases, without altering their transaction values.

In [24]:
# ============================================================
# Step 3 - Cell 1
# Load Step 2 mapping and immutable corrected transactions
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


STEP2_MAPPING_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step2_branded_beverages.csv"
)


STEP3_MAPPING_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step3_price_tier_metadata.csv"
)


STEP3_TIER_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step3_price_tier_product_audit.csv"
)


STEP3_PRICE_PERIOD_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step3_price_tier_price_periods.csv"
)


STEP3_PRICE_TRANSITION_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step3_price_change_transitions.csv"
)


STEP3_OTHER_DINNER_PRICE_CHANGES_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step3_other_dinner_price_changes.csv"
)


if not TRANSACTION_INPUT_FILE.exists():
    raise FileNotFoundError(
        "Corrected transaction file not found:\n"
        f"{TRANSACTION_INPUT_FILE}"
    )


if not STEP2_MAPPING_INPUT_FILE.exists():
    raise FileNotFoundError(
        "Completed Step 2 mapping not found:\n"
        f"{STEP2_MAPPING_INPUT_FILE}"
    )


step3_transactions = pd.read_csv(
    TRANSACTION_INPUT_FILE
)


step3_mapping = pd.read_csv(
    STEP2_MAPPING_INPUT_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


# Preserve source structure for integrity validation
step3_source_columns = (
    step3_transactions.columns.tolist()
)


# ------------------------------------------------------------
# Convert source columns safely
# ------------------------------------------------------------

step3_transactions["TransDate"] = pd.to_datetime(
    step3_transactions["TransDate"],
    errors="raise"
)


step3_transactions["Date"] = (
    pd.to_datetime(
        step3_transactions["Date"],
        errors="raise"
    )
    .dt.normalize()
)


step3_transactions["PLUCode"] = (
    pd.to_numeric(
        step3_transactions["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step3_transactions["UnitSold"] = (
    pd.to_numeric(
        step3_transactions["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


step3_transactions["TransValue"] = pd.to_numeric(
    step3_transactions["TransValue"],
    errors="raise"
)


step3_mapping["PLUCode"] = (
    pd.to_numeric(
        step3_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step3_mapping["TotalUnits"] = (
    pd.to_numeric(
        step3_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


# ------------------------------------------------------------
# Record immutable source totals
# ------------------------------------------------------------

step3_source_row_count = len(
    step3_transactions
)


step3_source_unit_total = int(
    step3_transactions["UnitSold"].sum()
)


step3_source_value_cents = int(
    (
        step3_transactions["TransValue"]
        * 100
    )
    .round()
    .sum()
)


# ------------------------------------------------------------
# Validate Step 2 state
# ------------------------------------------------------------

assert step3_source_row_count == 138_983


assert step3_source_unit_total == 141_480


assert step3_transactions[
    "PLUCode"
].nunique() == 236


assert len(step3_mapping) == 236


assert step3_mapping[
    "PLUCode"
].is_unique


assert step3_mapping[
    "CanonicalProductID_Step1"
].nunique() == 236


assert step3_mapping[
    "CanonicalProductID_Step2"
].nunique() == 228


assert int(
    step3_mapping["TotalUnits"].sum()
) == 141_480


print("Step 3 input files loaded and validated.")
print()
print(
    "Transaction rows:",
    f"{step3_source_row_count:,}"
)
print(
    "Transaction units:",
    f"{step3_source_unit_total:,}"
)
print(
    "Step 2 canonical identities:",
    step3_mapping[
        "CanonicalProductID_Step2"
    ].nunique()
)

Step 3 input files loaded and validated.

Transaction rows: 138,983
Transaction units: 141,480
Step 2 canonical identities: 228


In [25]:
# ============================================================
# Step 3 - Cell 2
# Derive observed per-unit prices
# ============================================================

assert step3_transactions[
    "UnitSold"
].gt(0).all(), (
    "UnitSold must be above zero before calculating prices."
)


step3_transactions[
    "ObservedUnitPrice_Step3"
] = (
    step3_transactions["TransValue"]
    / step3_transactions["UnitSold"]
).round(2)


# ------------------------------------------------------------
# Confirm that price × quantity reconstructs transaction value
# ------------------------------------------------------------

reconstructed_transaction_values = (
    step3_transactions[
        "ObservedUnitPrice_Step3"
    ]
    * step3_transactions["UnitSold"]
)


assert np.allclose(
    reconstructed_transaction_values,
    step3_transactions["TransValue"],
    atol=0.011
), (
    "At least one transaction value cannot be reconstructed "
    "from ObservedUnitPrice × UnitSold."
)


print("Observed unit prices calculated successfully.")
print()
print(
    "Rows:",
    f"{len(step3_transactions):,}"
)

Observed unit prices calculated successfully.

Rows: 138,983


In [26]:
# ============================================================
# Step 3 - Cell 3
# Audit all DINNER products and observed unit prices
# ============================================================

dinner_transactions_step3 = (
    step3_transactions.loc[
        step3_transactions[
            "GroupName"
        ].eq("DINNER")
    ]
    .copy()
)


dinner_product_price_audit = (
    dinner_transactions_step3
    .groupby(
        [
            "PLUCode",
            "PLUName",
        ],
        as_index=False
    )
    .agg(
        FirstObservedDate=(
            "Date",
            "min"
        ),
        LastObservedDate=(
            "Date",
            "max"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
        UniqueObservedPrices=(
            "ObservedUnitPrice_Step3",
            "nunique"
        ),
        MinimumObservedUnitPrice=(
            "ObservedUnitPrice_Step3",
            "min"
        ),
        MaximumObservedUnitPrice=(
            "ObservedUnitPrice_Step3",
            "max"
        ),
        ObservedPrices=(
            "ObservedUnitPrice_Step3",
            lambda values: ", ".join(
                f"{price:.2f}"
                for price in sorted(
                    values.unique()
                )
            )
        ),
    )
    .sort_values(
        [
            "FirstObservedDate",
            "PLUCode",
        ]
    )
    .reset_index(drop=True)
)


print("DINNER product-price audit created.")
print()
print(
    "Dinner PLUs:",
    dinner_product_price_audit[
        "PLUCode"
    ].nunique()
)

display(dinner_product_price_audit)

DINNER product-price audit created.

Dinner PLUs: 21


,PLUCode,PLUName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,UniqueObservedPrices,MinimumObservedUnitPrice,MaximumObservedUnitPrice,ObservedPrices
0,4241475,KIMBOX MAINS 1,2025-04-01,2025-10-13,1037,1037,1,5.0,5.0,5.00
1,4241476,KIMBOX MAINS 2,2025-04-01,2025-10-13,6575,6623,1,7.0,7.0,7.00
2,4241478,MAINS 1,2025-04-01,2025-10-09,1903,1903,1,5.0,5.0,5.00
3,4241479,MAINS 2,2025-04-01,2025-10-13,2265,2265,1,7.0,7.0,7.00
4,4241481,VEGT MAINS 1,2025-04-01,2025-09-16,878,878,1,5.0,5.0,5.00
5,4241482,VEGT MAINS 2,2025-04-02,2025-09-15,502,502,1,7.0,7.0,7.00
6,4241480,MAINS 3,2025-07-01,2025-08-01,37,474,1,9.0,9.0,9.00
7,4241483,VEGT MAINS 3,2025-07-01,2025-07-25,8,262,1,9.0,9.0,9.00
8,4241477,KIMBOX MAINS 3,2025-07-02,2025-09-02,5,5,1,9.0,9.0,9.00
9,42529,€5.00 DINNER,2025-10-14,2026-03-30,3312,3312,2,5.0,5.3,"5.00, 5.30"


In [27]:
# ============================================================
# Step 3 - Cell 4
# Define price-tier metadata
#
# Important:
# This assigns metadata only.
# It does not merge any PLUs.
# ============================================================

price_tier_metadata = [
    # --------------------------------------------------------
    # Current DINNER tiers
    # --------------------------------------------------------
    {
        "PLUCode": 42529,
        "ExpectedCorrectedName": "€5.00 DINNER",
        "TierProductFamily": "DINNER",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [5.00, 5.30],
    },
    {
        "PLUCode": 42530,
        "ExpectedCorrectedName": "€7 DINNER",
        "TierProductFamily": "DINNER",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [7.00, 7.30],
    },
    {
        "PLUCode": 42531,
        "ExpectedCorrectedName": "€9 DINNER",
        "TierProductFamily": "DINNER",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [9.00, 9.30],
    },

    # --------------------------------------------------------
    # Current KIMBOCK tiers
    # --------------------------------------------------------
    {
        "PLUCode": 42532,
        "ExpectedCorrectedName": "€5 KIMBOCK",
        "TierProductFamily": "KIMBOCK",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [5.00, 5.30],
    },
    {
        "PLUCode": 42533,
        "ExpectedCorrectedName": "€7 KIMBOCK",
        "TierProductFamily": "KIMBOCK",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [7.00, 7.30],
    },
    {
        "PLUCode": 42534,
        "ExpectedCorrectedName": "€9 KIMBOCK",
        "TierProductFamily": "KIMBOCK",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [9.00, 9.30],
    },

    # --------------------------------------------------------
    # Legacy KIMBOX tiers
    # --------------------------------------------------------
    {
        "PLUCode": 4241475,
        "ExpectedCorrectedName": "KIMBOX MAINS 1",
        "TierProductFamily": "KIMBOX",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": 1,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [5.00],
    },
    {
        "PLUCode": 4241476,
        "ExpectedCorrectedName": "KIMBOX MAINS 2",
        "TierProductFamily": "KIMBOX",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": 2,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [7.00],
    },
    {
        "PLUCode": 4241477,
        "ExpectedCorrectedName": "KIMBOX MAINS 3",
        "TierProductFamily": "KIMBOX",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": 3,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [9.00],
    },

    # --------------------------------------------------------
    # Legacy general MAINS tiers
    # --------------------------------------------------------
    {
        "PLUCode": 4241478,
        "ExpectedCorrectedName": "MAINS 1",
        "TierProductFamily": "MAINS",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": 1,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [5.00],
    },
    {
        "PLUCode": 4241479,
        "ExpectedCorrectedName": "MAINS 2",
        "TierProductFamily": "MAINS",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": 2,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [7.00],
    },
    {
        "PLUCode": 4241480,
        "ExpectedCorrectedName": "MAINS 3",
        "TierProductFamily": "MAINS",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": 3,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [9.00],
    },

    # --------------------------------------------------------
    # Legacy vegetarian MAINS tiers
    # --------------------------------------------------------
    {
        "PLUCode": 4241481,
        "ExpectedCorrectedName": "VEGT MAINS 1",
        "TierProductFamily": "VEGETARIAN MAINS",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": 1,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [5.00],
    },
    {
        "PLUCode": 4241482,
        "ExpectedCorrectedName": "VEGT MAINS 2",
        "TierProductFamily": "VEGETARIAN MAINS",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": 2,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [7.00],
    },
    {
        "PLUCode": 4241483,
        "ExpectedCorrectedName": "VEGT MAINS 3",
        "TierProductFamily": "VEGETARIAN MAINS",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": 3,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [9.00],
    },
]


price_tier_metadata_df = pd.DataFrame(
    price_tier_metadata
)


assert len(price_tier_metadata_df) == 15


assert price_tier_metadata_df[
    "PLUCode"
].is_unique


approved_price_tier_plu_codes = set(
    price_tier_metadata_df["PLUCode"]
)


print("Price-tier metadata defined.")
print()
print(
    "Price-tier PLUs:",
    len(price_tier_metadata_df)
)

display(
    price_tier_metadata_df[
        [
            "PLUCode",
            "ExpectedCorrectedName",
            "TierProductFamily",
            "NominalPriceTier",
            "LegacyMenuLevel",
            "MenuGeneration",
            "AllowedObservedPrices",
        ]
    ]
)

Price-tier metadata defined.

Price-tier PLUs: 15


,PLUCode,ExpectedCorrectedName,TierProductFamily,NominalPriceTier,LegacyMenuLevel,MenuGeneration,AllowedObservedPrices
0,42529,€5.00 DINNER,DINNER,5,<NA>,CURRENT_PRICE_LABELLED,"[5.0, 5.3]"
1,42530,€7 DINNER,DINNER,7,<NA>,CURRENT_PRICE_LABELLED,"[7.0, 7.3]"
2,42531,€9 DINNER,DINNER,9,<NA>,CURRENT_PRICE_LABELLED,"[9.0, 9.3]"
3,42532,€5 KIMBOCK,KIMBOCK,5,<NA>,CURRENT_PRICE_LABELLED,"[5.0, 5.3]"
4,42533,€7 KIMBOCK,KIMBOCK,7,<NA>,CURRENT_PRICE_LABELLED,"[7.0, 7.3]"
5,42534,€9 KIMBOCK,KIMBOCK,9,<NA>,CURRENT_PRICE_LABELLED,"[9.0, 9.3]"
6,4241475,KIMBOX MAINS 1,KIMBOX,5,1,LEGACY_LEVEL_LABELLED,[5.0]
7,4241476,KIMBOX MAINS 2,KIMBOX,7,2,LEGACY_LEVEL_LABELLED,[7.0]
8,4241477,KIMBOX MAINS 3,KIMBOX,9,3,LEGACY_LEVEL_LABELLED,[9.0]
9,4241478,MAINS 1,MAINS,5,1,LEGACY_LEVEL_LABELLED,[5.0]


In [28]:
# ============================================================
# Step 3 - Cell 5
# Preflight validation of all 15 tier products
# ============================================================

tier_transactions_step3 = (
    step3_transactions.loc[
        step3_transactions[
            "PLUCode"
        ].isin(approved_price_tier_plu_codes)
    ]
    .copy()
)


assert tier_transactions_step3[
    "PLUCode"
].nunique() == 15


assert len(tier_transactions_step3) == 32_733, (
    "Expected 32,733 transaction rows for the "
    "15 price-tier products."
)


assert int(
    tier_transactions_step3["UnitSold"].sum()
) == 33_552, (
    "Expected 33,552 units for the "
    "15 price-tier products."
)


assert tier_transactions_step3[
    "GroupName"
].eq("DINNER").all()


for tier_mapping in price_tier_metadata:

    plu_code = tier_mapping["PLUCode"]


    # --------------------------------------------------------
    # Validate Step 2 mapping name
    # --------------------------------------------------------

    mapping_mask = (
        step3_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(mapping_mask.sum()) == 1, (
        f"Expected one Step 2 mapping row for PLU {plu_code}."
    )


    actual_corrected_name = (
        step3_mapping.loc[
            mapping_mask,
            "PLUName_Corrected"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    expected_corrected_name = (
        tier_mapping[
            "ExpectedCorrectedName"
        ]
        .strip()
        .upper()
    )


    assert actual_corrected_name == (
        expected_corrected_name
    ), (
        f"Name mismatch for PLU {plu_code}.\n"
        f"Expected: {expected_corrected_name}\n"
        f"Found: {actual_corrected_name}"
    )


    # --------------------------------------------------------
    # Validate observed unit prices
    # --------------------------------------------------------

    product_transactions = (
        tier_transactions_step3.loc[
            tier_transactions_step3[
                "PLUCode"
            ].eq(plu_code)
        ]
    )


    actual_prices = {
        round(float(price), 2)
        for price in product_transactions[
            "ObservedUnitPrice_Step3"
        ].unique()
    }


    allowed_prices = {
        round(float(price), 2)
        for price in tier_mapping[
            "AllowedObservedPrices"
        ]
    }


    assert actual_prices == allowed_prices, (
        f"Observed-price mismatch for PLU {plu_code}.\n"
        f"Expected: {sorted(allowed_prices)}\n"
        f"Found: {sorted(actual_prices)}"
    )


print("All 15 price-tier products passed preflight.")
print()
print(
    "Tier transaction rows:",
    f"{len(tier_transactions_step3):,}"
)
print(
    "Tier units:",
    f"{int(tier_transactions_step3['UnitSold'].sum()):,}"
)

All 15 price-tier products passed preflight.

Tier transaction rows: 32,733
Tier units: 33,552
